In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
import random
import numpy as np

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))

# from model import SeqNN
from akita_model.model import SeqNN

In [ ]:
# FOLD = 0

# fold 0 - test
# fold 1 - valid
# folds 2-7 - train

In [ ]:
TARGET_C = -0.5

# TARGET_C = -10.0

In [ ]:
# df = pd.read_csv(f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/results_repeated/fold{FOLD}_with_positions_steps_results_mod.tsv", sep="\t")
df = pd.read_csv(f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/results_repeated/only_successful_seqs_with_results.tsv", sep="\t")

In [ ]:
len(df)

In [ ]:
df.columns

In [ ]:
class OriginalDataset(Dataset):
    def __init__(self, coord_df, init_seq_path):
        self.coords = coord_df
        self.init_seq_path = init_seq_path
    
    def __len__(self):
        return len(self.coords)

    def __getitem__(self, idx):
        row = self.coords.iloc[idx]
        chrom = row["chrom"]
        start = row["centered_start"]
        end = row["centered_end"]
        fold = row["fold"]
        
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        
        X = torch.load(f"{self.init_seq_path}/fold{fold}/{chrom}_{start}_{end}_X.pt", weights_only=True, map_location=device)
        X = X.squeeze(0)
        return X

In [ ]:
# class BoundaryGenerationDataset(Dataset):
#     def __init__(self, coord_df, init_seq_path, slice_path, slice=256, cropping=64, bin_size=2048):
#         self.coords = coord_df
#         self.init_seq_path = init_seq_path
#         self.slice_path = slice_path
#         self.slice = slice
#         self.cropping = cropping
#         self.bin_size = bin_size
        
#     def __len__(self):
#         return len(self.coords)

#     def __getitem__(self, idx):
#         row = self.coords.iloc[idx]
#         chrom = row["chrom"]
#         start = row["centered_start"]
#         end = row["centered_end"]
#         fold = row["fold"]
        
#         device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        
#         X = torch.load(f"{self.init_seq_path}/fold{fold}/{chrom}_{start}_{end}_X.pt", weights_only=True, map_location=device)
#         slice = torch.load(f"{self.slice_path}/fold{fold}/{chrom}_{start}_{end}_slice.pt", weights_only=True, map_location=device)
        
#         edit_start = (self.slice + self.cropping) * self.bin_size
#         edit_end = edit_start + self.bin_size
        
#         editedX = X.clone()
#         editedX[:,:, edit_start:edit_end] = slice
        
#         editedX = editedX.squeeze(0)
        
#         return editedX

In [ ]:
# ### CONTROL: REPLACING G AND C HOMOPOLYMERS IN ORIGINAL SEQUENCES

# import re
# import torch

# class OriginalDataset(Dataset):
#     def __init__(self, coord_df, init_seq_path, slice=256, cropping=64, bin_size=2048, min_homopolymer_length=4):
#         """
#         Dataset that replaces homopolymers in ORIGINAL sequences (before optimization).
        
#         This is a CONTROL experiment to test:
#         - Do homopolymers suppress boundaries in natural sequences?
#         - Or is this effect specific to optimized sequences?
        
#         Args:
#             coord_df: DataFrame with sequence coordinates
#             init_seq_path: Path to initial sequences
#             slice: Slice parameter (default: 256)
#             cropping: Cropping parameter (default: 64)
#             bin_size: Bin size (default: 2048)
#             min_homopolymer_length: Minimum length of homopolymers to replace (default: 4)
#         """
#         self.coords = coord_df
#         self.init_seq_path = init_seq_path
#         self.slice = slice
#         self.cropping = cropping
#         self.bin_size = bin_size
#         self.min_homopolymer_length = min_homopolymer_length
    
#     def __len__(self):
#         return len(self.coords)
    
#     def sequence_to_tensor(self, sequence, device):
#         """Convert DNA sequence string to one-hot tensor."""
#         mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
#         tensor = torch.zeros(4, len(sequence), device=device)
        
#         for i, base in enumerate(sequence):
#             if base in mapping:
#                 tensor[mapping[base], i] = 1
        
#         return tensor
    
#     def tensor_to_sequence(self, seq_tensor):
#         """Convert one-hot tensor to DNA sequence string."""
#         if seq_tensor.dim() == 3:
#             seq_tensor = seq_tensor.squeeze(0)
        
#         nucleotides = ['A', 'C', 'G', 'T']
#         indices = torch.argmax(seq_tensor, dim=0)
#         sequence = ''.join([nucleotides[i] for i in indices])
#         return sequence
    
#     def replace_gc_homopolymers_in_slice(self, full_tensor, min_length=None):
#         """
#         Replace G and C homopolymers in the central slice only.
        
#         This replaces homopolymers in the same region that was optimized
#         in the boundary suppression experiments.
        
#         Args:
#             full_tensor: [4, sequence_length] tensor (full sequence)
#             min_length: minimum homopolymer length to replace
#                        If None, uses self.min_homopolymer_length
        
#         Returns:
#             Modified tensor with homopolymers replaced in slice region
#         """
#         device = full_tensor.device
        
#         # Use instance min_length if not specified
#         if min_length is None:
#             min_length = self.min_homopolymer_length
        
#         # Define slice boundaries (same as in optimization)
#         edit_start = (self.slice + self.cropping) * self.bin_size
#         edit_end = edit_start + self.bin_size
        
#         # Extract the slice
#         slice_tensor = full_tensor[:, edit_start:edit_end]
        
#         # Convert slice to sequence
#         sequence = self.tensor_to_sequence(slice_tensor)
        
#         # Find and replace C homopolymers (CCCC+ -> GCGC...)
#         pattern_c = f'C{{{min_length},}}'
#         for match in re.finditer(pattern_c, sequence):
#             length = len(match.group())
#             replacement = 'GC' * (length // 2) + ('C' if length % 2 == 1 else '')
#             sequence = sequence[:match.start()] + replacement + sequence[match.end():]
        
#         # Find and replace G homopolymers (GGGG+ -> CGCG...)
#         pattern_g = f'G{{{min_length},}}'
#         for match in re.finditer(pattern_g, sequence):
#             length = len(match.group())
#             replacement = 'CG' * (length // 2) + ('G' if length % 2 == 1 else '')
#             sequence = sequence[:match.start()] + replacement + sequence[match.end():]
        
#         # Convert back to tensor
#         modified_slice = self.sequence_to_tensor(sequence, device)
        
#         # Replace slice in full tensor
#         modified_tensor = full_tensor.clone()
#         modified_tensor[:, edit_start:edit_end] = modified_slice
        
#         return modified_tensor

#     def __getitem__(self, idx):
#         row = self.coords.iloc[idx]
#         chrom = row["chrom"]
#         start = row["centered_start"]
#         end = row["centered_end"]

#         device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        
#         X = torch.load(f"{self.init_seq_path}{chrom}_{start}_{end}_X.pt", weights_only=True, map_location=device)
#         X = X.squeeze(0)  # [4, sequence_length]
        
#         # Replace homopolymers in the central slice
#         X_modified = self.replace_gc_homopolymers_in_slice(X)
        
#         if idx < 3:
#             # Debug output for first few sequences
#             edit_start = (self.slice + self.cropping) * self.bin_size
#             edit_end = edit_start + self.bin_size
            
#             orig_slice = X[:, edit_start:edit_end]
#             mod_slice = X_modified[:, edit_start:edit_end]
            
#             orig_seq = self.tensor_to_sequence(orig_slice)
#             mod_seq = self.tensor_to_sequence(mod_slice)
            
#             # Count homopolymers
#             orig_c_runs = len(re.findall(f'C{{{self.min_homopolymer_length},}}', orig_seq))
#             orig_g_runs = len(re.findall(f'G{{{self.min_homopolymer_length},}}', orig_seq))
#             mod_c_runs = len(re.findall(f'C{{{self.min_homopolymer_length},}}', mod_seq))
#             mod_g_runs = len(re.findall(f'G{{{self.min_homopolymer_length},}}', mod_seq))
            
#             # Count GC repeats
#             orig_gcgc = len(re.findall(r'GCGC', orig_seq))
#             orig_cgcg = len(re.findall(r'CGCG', orig_seq))
#             mod_gcgc = len(re.findall(r'GCGC', mod_seq))
#             mod_cgcg = len(re.findall(r'CGCG', mod_seq))
            
#             # Verify GC content
#             orig_gc = (orig_seq.count('G') + orig_seq.count('C')) / len(orig_seq)
#             mod_gc = (mod_seq.count('G') + mod_seq.count('C')) / len(mod_seq)
            
#             print(f"\nORIGINAL sequence {idx} (min_length={self.min_homopolymer_length}):")
#             print(f"  Homopolymers (≥{self.min_homopolymer_length}bp):")
#             print(f"    C-runs: {orig_c_runs} → {mod_c_runs} ({mod_c_runs - orig_c_runs:+d})")
#             print(f"    G-runs: {orig_g_runs} → {mod_g_runs} ({mod_g_runs - orig_g_runs:+d})")
#             print(f"    Total replaced: {orig_c_runs + orig_g_runs - mod_c_runs - mod_g_runs}")
            
#             print(f"  GC repeats:")
#             print(f"    GCGC: {orig_gcgc} → {mod_gcgc} ({mod_gcgc - orig_gcgc:+d})")
#             print(f"    CGCG: {orig_cgcg} → {mod_cgcg} ({mod_cgcg - orig_cgcg:+d})")
            
#             print(f"  GC content: {orig_gc:.4f} → {mod_gc:.4f} (diff: {mod_gc - orig_gc:.6f})")
        
#         return X_modified

In [ ]:
# ### SHUFLING THE ENTIRE SLICE   

# class BoundaryGenerationDataset(Dataset):
#     def __init__(self, coord_df, init_seq_path, slice_path, slice=256, cropping=64, bin_size=2048, seed=42):
#         self.coords = coord_df
#         self.init_seq_path = init_seq_path
#         self.slice_path = slice_path
#         self.slice = slice
#         self.cropping = cropping
#         self.bin_size = bin_size
#         self.seed = seed
        
#     def __len__(self):
#         return len(self.coords)
    
#     def shuffle_nucleotides_in_slice(self, slice_tensor, seed):
#         """
#         Shuffle nucleotides within the slice to destroy motifs.
#         slice_tensor shape: [1, 4, 2048]
#         """
#         # Set seed for this specific slice
#         torch.manual_seed(seed)
        
#         # Convert one-hot to indices (which nucleotide at each position)
#         indices = torch.argmax(slice_tensor[0], dim=0)  # [2048]
        
#         # Shuffle the indices
#         perm = torch.randperm(indices.shape[0])
#         shuffled_indices = indices[perm]
        
#         # Convert back to one-hot
#         shuffled_slice = torch.zeros_like(slice_tensor)
#         for i, idx in enumerate(shuffled_indices):
#             shuffled_slice[0, idx, i] = 1
        
#         return shuffled_slice

#     def __getitem__(self, idx):
#         row = self.coords.iloc[idx]
#         chrom = row["chrom"]
#         start = row["centered_start"]
#         end = row["centered_end"]
#         fold = row["fold"]
        
#         device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        
#         X = torch.load(
#             f"{self.init_seq_path}/fold{fold}/{chrom}_{start}_{end}_X.pt",
#             weights_only=True,
#             map_location=device
#         )
#         slice_edited = torch.load(
#             f"{self.slice_path}/fold{fold}/{chrom}_{start}_{end}_slice.pt",
#             weights_only=True,
#             map_location=device
#         )

#         # Shuffle nucleotides in the entire slice
#         slice_shuffled = self.shuffle_nucleotides_in_slice(slice_edited, self.seed + idx)

#         edit_start = (self.slice + self.cropping) * self.bin_size
#         edit_end = edit_start + self.bin_size
        
#         editedX = X.clone()
#         editedX[:, :, edit_start:edit_end] = slice_shuffled
        
#         editedX = editedX.squeeze(0)
        
#         return editedX

In [ ]:
# ### SHUFFLING EEVERYTHING BUT NOT THE ORIGINAL CTCFS (WITH FLANKS)

# class BoundaryGenerationDataset(Dataset):
#     def __init__(self, coord_df, init_seq_path, slice_path, slice=256, cropping=64, bin_size=2048, seed=42, flank_size=15):
#         self.coords = coord_df
#         self.init_seq_path = init_seq_path
#         self.slice_path = slice_path
#         self.slice = slice
#         self.cropping = cropping
#         self.bin_size = bin_size
#         self.seed = seed
#         self.flank_size = flank_size
        
#     def __len__(self):
#         return len(self.coords)
    
#     def shuffle_nucleotides_except_regions(self, slice_tensor, protected_regions, seed):
#         """
#         Shuffle nucleotides in the slice EXCEPT for protected regions (original CTCFs + flanks).
#         slice_tensor shape: [1, 4, 2048]
#         protected_regions: list of (start, end) tuples to preserve
#         """
#         # Set seed
#         torch.manual_seed(seed)
        
#         # Create a mask of positions to shuffle (True = shuffle, False = keep)
#         length = slice_tensor.shape[2]
#         shuffle_mask = torch.ones(length, dtype=torch.bool)
        
#         # Mark protected regions as False (don't shuffle)
#         for start, end in protected_regions:
#             if 0 <= start < length and 0 <= end <= length:
#                 shuffle_mask[start:end] = False
        
#         # Get indices of positions to shuffle
#         shuffle_positions = torch.where(shuffle_mask)[0]
        
#         if len(shuffle_positions) == 0:
#             # Nothing to shuffle
#             return slice_tensor.clone()
        
#         # Extract nucleotides at shuffleable positions
#         indices = torch.argmax(slice_tensor[0], dim=0)  # [2048]
#         shuffleable_nucleotides = indices[shuffle_positions]
        
#         # Shuffle them
#         perm = torch.randperm(len(shuffleable_nucleotides))
#         shuffled_nucleotides = shuffleable_nucleotides[perm]
        
#         # Create output tensor (start with original)
#         shuffled_slice = slice_tensor.clone()
        
#         # Clear the shuffleable positions
#         shuffled_slice[:, :, shuffle_positions] = 0
        
#         # Put shuffled nucleotides back
#         for i, pos in enumerate(shuffle_positions):
#             nuc_idx = shuffled_nucleotides[i]
#             shuffled_slice[0, nuc_idx, pos] = 1
        
#         return shuffled_slice

#     def __getitem__(self, idx):
#         row = self.coords.iloc[idx]
#         chrom = row["chrom"]
#         start = row["centered_start"]
#         end = row["centered_end"]
        
#         device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        
#         X = torch.load(
#             f"{self.init_seq_path}{chrom}_{start}_{end}_X.pt",
#             weights_only=True,
#             map_location=device
#         )
#         slice_edited = torch.load(
#             f"{self.slice_path}{chrom}_{start}_{end}_slice.pt",
#             weights_only=True,
#             map_location=device
#         )

#         # Get original CTCF positions to protect (with flanks)
#         protected_regions = []
#         if 'orig_CTCFs_coord' in row and row['orig_CTCFs_coord']:
#             orig_ctcfs = row['orig_CTCFs_coord']
            
#             # Convert from string to set if needed
#             if isinstance(orig_ctcfs, str):
#                 import ast
#                 orig_ctcfs = ast.literal_eval(orig_ctcfs)
            
#             # Collect regions to protect with ±15bp flanks
#             for ctcf_start, ctcf_end, strand in orig_ctcfs:
#                 # Add flanks
#                 protected_start = max(0, ctcf_start - self.flank_size)
#                 protected_end = min(self.bin_size, ctcf_end + self.flank_size)
                
#                 # Only protect if within bounds
#                 if 0 <= protected_start < self.bin_size and 0 <= protected_end <= self.bin_size:
#                     protected_regions.append((protected_start, protected_end))
        
#         if idx < 3:
#             print(f"\nSequence {idx}: Protecting {len(protected_regions)} original CTCF regions with ±{self.flank_size}bp flanks:")
#             for start, end in protected_regions:
#                 print(f"  {start}-{end} (length={end-start}bp)")
        
#         # Shuffle everything except original CTCFs + flanks
#         slice_shuffled = self.shuffle_nucleotides_except_regions(
#             slice_edited, 
#             protected_regions, 
#             self.seed + idx
#         )

#         edit_start = (self.slice + self.cropping) * self.bin_size
#         edit_end = edit_start + self.bin_size
        
#         editedX = X.clone()
#         editedX[:, :, edit_start:edit_end] = slice_shuffled
        
#         editedX = editedX.squeeze(0)
        
#         return editedX

In [ ]:
# ### SHUFFLING EEVERYTHING BUT NOT THE NEW CTCFS (WITH FLANKS)

# class BoundaryGenerationDataset(Dataset):
#     def __init__(self, coord_df, init_seq_path, slice_path, slice=256, cropping=64, bin_size=2048, seed=42, flank_size=15):
#         self.coords = coord_df
#         self.init_seq_path = init_seq_path
#         self.slice_path = slice_path
#         self.slice = slice
#         self.cropping = cropping
#         self.bin_size = bin_size
#         self.seed = seed
#         self.flank_size = flank_size
        
#     def __len__(self):
#         return len(self.coords)
    
#     def shuffle_nucleotides_except_regions(self, slice_tensor, protected_regions, seed):
#         """
#         Shuffle nucleotides in the slice EXCEPT for protected regions (original CTCFs + flanks).
#         slice_tensor shape: [1, 4, 2048]
#         protected_regions: list of (start, end) tuples to preserve
#         """
#         # Set seed
#         torch.manual_seed(seed)
        
#         # Create a mask of positions to shuffle (True = shuffle, False = keep)
#         length = slice_tensor.shape[2]
#         shuffle_mask = torch.ones(length, dtype=torch.bool)
        
#         # Mark protected regions as False (don't shuffle)
#         for start, end in protected_regions:
#             if 0 <= start < length and 0 <= end <= length:
#                 shuffle_mask[start:end] = False
        
#         # Get indices of positions to shuffle
#         shuffle_positions = torch.where(shuffle_mask)[0]
        
#         if len(shuffle_positions) == 0:
#             # Nothing to shuffle
#             return slice_tensor.clone()
        
#         # Extract nucleotides at shuffleable positions
#         indices = torch.argmax(slice_tensor[0], dim=0)  # [2048]
#         shuffleable_nucleotides = indices[shuffle_positions]
        
#         # Shuffle them
#         perm = torch.randperm(len(shuffleable_nucleotides))
#         shuffled_nucleotides = shuffleable_nucleotides[perm]
        
#         # Create output tensor (start with original)
#         shuffled_slice = slice_tensor.clone()
        
#         # Clear the shuffleable positions
#         shuffled_slice[:, :, shuffle_positions] = 0
        
#         # Put shuffled nucleotides back
#         for i, pos in enumerate(shuffle_positions):
#             nuc_idx = shuffled_nucleotides[i]
#             shuffled_slice[0, nuc_idx, pos] = 1
        
#         return shuffled_slice

#     def __getitem__(self, idx):
#         row = self.coords.iloc[idx]
#         chrom = row["chrom"]
#         start = row["centered_start"]
#         end = row["centered_end"]
        
#         device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        
#         X = torch.load(
#             f"{self.init_seq_path}{chrom}_{start}_{end}_X.pt",
#             weights_only=True,
#             map_location=device
#         )
#         slice_edited = torch.load(
#             f"{self.slice_path}{chrom}_{start}_{end}_slice.pt",
#             weights_only=True,
#             map_location=device
#         )

#         # Get original CTCF positions to protect (with flanks)
#         protected_regions = []
#         if 'new_CTCFs_coord' in row and row['new_CTCFs_coord']:
#             orig_ctcfs = row['new_CTCFs_coord']
            
#             # Convert from string to set if needed
#             if isinstance(orig_ctcfs, str):
#                 import ast
#                 orig_ctcfs = ast.literal_eval(orig_ctcfs)
            
#             # Collect regions to protect with ±15bp flanks
#             for ctcf_start, ctcf_end, strand in orig_ctcfs:
#                 # Add flanks
#                 protected_start = max(0, ctcf_start - self.flank_size)
#                 protected_end = min(self.bin_size, ctcf_end + self.flank_size)
                
#                 # Only protect if within bounds
#                 if 0 <= protected_start < self.bin_size and 0 <= protected_end <= self.bin_size:
#                     protected_regions.append((protected_start, protected_end))
        
#         if idx < 3:
#             print(f"\nSequence {idx}: Protecting {len(protected_regions)} new CTCF regions with ±{self.flank_size}bp flanks:")
#             for start, end in protected_regions:
#                 print(f"  {start}-{end} (length={end-start}bp)")
        
#         # Shuffle everything except original CTCFs + flanks
#         slice_shuffled = self.shuffle_nucleotides_except_regions(
#             slice_edited, 
#             protected_regions, 
#             self.seed + idx
#         )

#         edit_start = (self.slice + self.cropping) * self.bin_size
#         edit_end = edit_start + self.bin_size
        
#         editedX = X.clone()
#         editedX[:, :, edit_start:edit_end] = slice_shuffled
        
#         editedX = editedX.squeeze(0)
        
#         return editedX

In [ ]:
# ### SHUFFLING EVERYTHING BUT NOT THE ORIGINAL AND NEWLY ADDED CTCFS (WITH FLANKS)

# class BoundaryGenerationDataset(Dataset):
#     def __init__(self, coord_df, init_seq_path, slice_path, slice=256, cropping=64, bin_size=2048, seed=42, flank_size=15):
#         self.coords = coord_df
#         self.init_seq_path = init_seq_path
#         self.slice_path = slice_path
#         self.slice = slice
#         self.cropping = cropping
#         self.bin_size = bin_size
#         self.seed = seed
#         self.flank_size = flank_size
        
#     def __len__(self):
#         return len(self.coords)
    
#     def shuffle_nucleotides_except_regions(self, slice_tensor, protected_regions, seed):
#         """
#         Shuffle nucleotides in the slice EXCEPT for protected regions (CTCFs + flanks).
#         slice_tensor shape: [1, 4, 2048]
#         protected_regions: list of (start, end) tuples to preserve
#         """
#         # Set seed
#         torch.manual_seed(seed)
        
#         # Create a mask of positions to shuffle (True = shuffle, False = keep)
#         length = slice_tensor.shape[2]
#         shuffle_mask = torch.ones(length, dtype=torch.bool)
        
#         # Mark protected regions as False (don't shuffle)
#         for start, end in protected_regions:
#             if 0 <= start < length and 0 <= end <= length:
#                 shuffle_mask[start:end] = False
        
#         # Get indices of positions to shuffle
#         shuffle_positions = torch.where(shuffle_mask)[0]
        
#         if len(shuffle_positions) == 0:
#             # Nothing to shuffle
#             return slice_tensor.clone()
        
#         # Extract nucleotides at shuffleable positions
#         indices = torch.argmax(slice_tensor[0], dim=0)  # [2048]
#         shuffleable_nucleotides = indices[shuffle_positions]
        
#         # Shuffle them
#         perm = torch.randperm(len(shuffleable_nucleotides))
#         shuffled_nucleotides = shuffleable_nucleotides[perm]
        
#         # Create output tensor (start with original)
#         shuffled_slice = slice_tensor.clone()
        
#         # Clear the shuffleable positions
#         shuffled_slice[:, :, shuffle_positions] = 0
        
#         # Put shuffled nucleotides back
#         for i, pos in enumerate(shuffle_positions):
#             nuc_idx = shuffled_nucleotides[i]
#             shuffled_slice[0, nuc_idx, pos] = 1
        
#         return shuffled_slice

#     def __getitem__(self, idx):
#         row = self.coords.iloc[idx]
#         chrom = row["chrom"]
#         start = row["centered_start"]
#         end = row["centered_end"]
        
#         device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        
#         X = torch.load(
#             f"{self.init_seq_path}{chrom}_{start}_{end}_X.pt",
#             weights_only=True,
#             map_location=device
#         )
#         slice_edited = torch.load(
#             f"{self.slice_path}{chrom}_{start}_{end}_slice.pt",
#             weights_only=True,
#             map_location=device
#         )

#         # Get BOTH original AND new CTCF positions to protect (with flanks)
#         protected_regions = []
        
#         # Protect original CTCFs
#         if 'orig_CTCFs_coord' in row and row['orig_CTCFs_coord']:
#             orig_ctcfs = row['orig_CTCFs_coord']
            
#             # Convert from string to set if needed
#             if isinstance(orig_ctcfs, str):
#                 import ast
#                 orig_ctcfs = ast.literal_eval(orig_ctcfs)
            
#             # Collect regions to protect with ±15bp flanks
#             for ctcf_start, ctcf_end, strand in orig_ctcfs:
#                 # Add flanks
#                 protected_start = max(0, ctcf_start - self.flank_size)
#                 protected_end = min(self.bin_size, ctcf_end + self.flank_size)
                
#                 # Only protect if within bounds
#                 if 0 <= protected_start < self.bin_size and 0 <= protected_end <= self.bin_size:
#                     protected_regions.append((protected_start, protected_end))
        
#         # Protect new CTCFs
#         if 'new_CTCFs_coord' in row and row['new_CTCFs_coord']:
#             new_ctcfs = row['new_CTCFs_coord']
            
#             # Convert from string to set if needed
#             if isinstance(new_ctcfs, str):
#                 import ast
#                 new_ctcfs = ast.literal_eval(new_ctcfs)
            
#             # Collect regions to protect with ±15bp flanks
#             for ctcf_start, ctcf_end, strand in new_ctcfs:
#                 # Add flanks
#                 protected_start = max(0, ctcf_start - self.flank_size)
#                 protected_end = min(self.bin_size, ctcf_end + self.flank_size)
                
#                 # Only protect if within bounds
#                 if 0 <= protected_start < self.bin_size and 0 <= protected_end <= self.bin_size:
#                     protected_regions.append((protected_start, protected_end))
        
#         if idx < 3:
#             print(f"\nSequence {idx}: Protecting {len(protected_regions)} CTCF regions (orig + new) with ±{self.flank_size}bp flanks")
        
#         # Shuffle everything except ALL CTCFs + flanks
#         slice_shuffled = self.shuffle_nucleotides_except_regions(
#             slice_edited, 
#             protected_regions, 
#             self.seed + idx
#         )

#         edit_start = (self.slice + self.cropping) * self.bin_size
#         edit_end = edit_start + self.bin_size
        
#         editedX = X.clone()
#         editedX[:, :, edit_start:edit_end] = slice_shuffled
        
#         editedX = editedX.squeeze(0)
        
#         return editedX

In [ ]:
# ### SHUFFLING ONLY NEW CTCFS

# class BoundaryGenerationDataset(Dataset):
#     def __init__(self, coord_df, init_seq_path, slice_path, slice=256, cropping=64, bin_size=2048, seed=42):
#         self.coords = coord_df
#         self.init_seq_path = init_seq_path
#         self.slice_path = slice_path
#         self.slice = slice
#         self.cropping = cropping
#         self.bin_size = bin_size
#         self.seed = seed
        
#     def __len__(self):
#         return len(self.coords)
    
#     def shuffle_sequence_region(self, seq_tensor, start, end, seed):
#         """
#         Shuffle nucleotides within a specific region.
#         seq_tensor shape: [1, 4, seq_len]
#         """
#         # Set seed
#         torch.manual_seed(seed)
        
#         # Extract the region
#         region = seq_tensor[:, :, start:end].clone()  # [1, 4, length]
#         length = end - start
        
#         # Convert one-hot to indices
#         indices = torch.argmax(region[0], dim=0)  # [length]
        
#         # Shuffle
#         perm = torch.randperm(length)
#         shuffled_indices = indices[perm]
        
#         # Convert back to one-hot
#         shuffled_region = torch.zeros_like(region)
#         for i, idx in enumerate(shuffled_indices):
#             shuffled_region[0, idx, i] = 1
        
#         # Put it back
#         seq_tensor[:, :, start:end] = shuffled_region
        
#         return seq_tensor

#     def __getitem__(self, idx):
#         row = self.coords.iloc[idx]
#         chrom = row["chrom"]
#         start = row["centered_start"]
#         end = row["centered_end"]
#         fold = row["fold"]
        
#         device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        
#         X = torch.load(
#             f"{self.init_seq_path}/fold{fold}/{chrom}_{start}_{end}_X.pt",
#             weights_only=True,
#             map_location=device
#         )
#         slice_edited = torch.load(
#             f"{self.slice_path}/fold{fold}/{chrom}_{start}_{end}_slice.pt",
#             weights_only=True,
#             map_location=device
#         )

#         # Clone the slice
#         modified_slice = slice_edited.clone()
        
#         # Shuffle ONLY the new CTCFs
#         if 'new_CTCFs_coord' in row and row['new_CTCFs_coord']:
#             new_ctcfs = row['new_CTCFs_coord']
            
#             # Convert from string to set if needed
#             if isinstance(new_ctcfs, str):
#                 import ast
#                 new_ctcfs = ast.literal_eval(new_ctcfs)
            
#             # Filter CTCFs within the editable region (0-2048)
#             valid_ctcfs = [(s, e, strand) for s, e, strand in new_ctcfs 
#                           if 0 <= s < self.bin_size and 0 <= e <= self.bin_size]
            
#             if valid_ctcfs:
#                 # Set seed for reproducibility
#                 torch.manual_seed(self.seed + idx)
                
#                 if idx < 3:
#                     print(f"\nSequence {idx}: Shuffling {len(valid_ctcfs)} new CTCF motifs only")
                
#                 # Shuffle each new CTCF motif
#                 for ctcf_idx, (ctcf_start, ctcf_end, strand) in enumerate(valid_ctcfs):
#                     if idx < 3:
#                         print(f"  Shuffling new CTCF at {ctcf_start}-{ctcf_end} (strand={strand})")
                    
#                     # Shuffle this CTCF with a unique seed
#                     modified_slice = self.shuffle_sequence_region(
#                         modified_slice, 
#                         ctcf_start, 
#                         ctcf_end, 
#                         self.seed + idx + ctcf_idx
#                     )

#         # Insert the modified slice into X
#         edit_start = (self.slice + self.cropping) * self.bin_size
#         edit_end = edit_start + self.bin_size
        
#         editedX = X.clone()
#         editedX[:, :, edit_start:edit_end] = modified_slice
        
#         editedX = editedX.squeeze(0)
        
#         return editedX

In [ ]:
# ### SHUFFLING ONLY ORIGINAL CTCFS

# class BoundaryGenerationDataset(Dataset):
#     def __init__(self, coord_df, init_seq_path, slice_path, slice=256, cropping=64, bin_size=2048, seed=42):
#         self.coords = coord_df
#         self.init_seq_path = init_seq_path
#         self.slice_path = slice_path
#         self.slice = slice
#         self.cropping = cropping
#         self.bin_size = bin_size
#         self.seed = seed
        
#     def __len__(self):
#         return len(self.coords)
    
#     def shuffle_sequence_region(self, seq_tensor, start, end, seed):
#         """
#         Shuffle nucleotides within a specific region.
#         seq_tensor shape: [1, 4, seq_len]
#         """
#         # Set seed
#         torch.manual_seed(seed)
        
#         # Extract the region
#         region = seq_tensor[:, :, start:end].clone()  # [1, 4, length]
#         length = end - start
        
#         # Convert one-hot to indices
#         indices = torch.argmax(region[0], dim=0)  # [length]
        
#         # Shuffle
#         perm = torch.randperm(length)
#         shuffled_indices = indices[perm]
        
#         # Convert back to one-hot
#         shuffled_region = torch.zeros_like(region)
#         for i, idx in enumerate(shuffled_indices):
#             shuffled_region[0, idx, i] = 1
        
#         # Put it back
#         seq_tensor[:, :, start:end] = shuffled_region
        
#         return seq_tensor

#     def __getitem__(self, idx):
#         row = self.coords.iloc[idx]
#         chrom = row["chrom"]
#         start = row["centered_start"]
#         end = row["centered_end"]
        
#         device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        
#         X = torch.load(
#             f"{self.init_seq_path}{chrom}_{start}_{end}_X.pt",
#             weights_only=True,
#             map_location=device
#         )
#         slice_edited = torch.load(
#             f"{self.slice_path}{chrom}_{start}_{end}_slice.pt",
#             weights_only=True,
#             map_location=device
#         )

#         # Clone the slice
#         modified_slice = slice_edited.clone()
        
#         # Shuffle ONLY the new CTCFs
#         if 'orig_CTCFs_coord' in row and row['orig_CTCFs_coord']:
#             new_ctcfs = row['orig_CTCFs_coord']
            
#             # Convert from string to set if needed
#             if isinstance(new_ctcfs, str):
#                 import ast
#                 new_ctcfs = ast.literal_eval(new_ctcfs)
            
#             # Filter CTCFs within the editable region (0-2048)
#             valid_ctcfs = [(s, e, strand) for s, e, strand in new_ctcfs 
#                           if 0 <= s < self.bin_size and 0 <= e <= self.bin_size]
            
#             if valid_ctcfs:
#                 # Set seed for reproducibility
#                 torch.manual_seed(self.seed + idx)
                
#                 if idx < 3:
#                     print(f"\nSequence {idx}: Shuffling {len(valid_ctcfs)} new CTCF motifs only")
                
#                 # Shuffle each new CTCF motif
#                 for ctcf_idx, (ctcf_start, ctcf_end, strand) in enumerate(valid_ctcfs):
#                     if idx < 3:
#                         print(f"  Shuffling new CTCF at {ctcf_start}-{ctcf_end} (strand={strand})")
                    
#                     # Shuffle this CTCF with a unique seed
#                     modified_slice = self.shuffle_sequence_region(
#                         modified_slice, 
#                         ctcf_start, 
#                         ctcf_end, 
#                         self.seed + idx + ctcf_idx
#                     )

#         # Insert the modified slice into X
#         edit_start = (self.slice + self.cropping) * self.bin_size
#         edit_end = edit_start + self.bin_size
        
#         editedX = X.clone()
#         editedX[:, :, edit_start:edit_end] = modified_slice
        
#         editedX = editedX.squeeze(0)
        
#         return editedX

In [ ]:
# ### INSERT CTCFS WITHOUT THE EDITED SLICE

# class BoundaryGenerationDataset(Dataset):
#     def __init__(self, coord_df, init_seq_path, slice_path, slice=256, cropping=64, bin_size=2048, flank_size=15):
#         self.coords = coord_df
#         self.init_seq_path = init_seq_path
#         self.slice_path = slice_path
#         self.slice = slice
#         self.cropping = cropping
#         self.bin_size = bin_size
#         self.flank_size = flank_size
        
#     def __len__(self):
#         return len(self.coords)

#     def __getitem__(self, idx):
#         row = self.coords.iloc[idx]
#         chrom = row["chrom"]
#         start = row["centered_start"]
#         end = row["centered_end"]
        
#         device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        
#         # Load original sequence
#         X = torch.load(
#             f"{self.init_seq_path}{chrom}_{start}_{end}_X.pt",
#             weights_only=True,
#             map_location=device
#         )
        
#         # Load edited slice
#         slice_edited = torch.load(
#             f"{self.slice_path}{chrom}_{start}_{end}_slice.pt",
#             weights_only=True,
#             map_location=device
#         )
        
#         edit_start = (self.slice + self.cropping) * self.bin_size
#         edit_end = edit_start + self.bin_size
        
#         # Start with the ORIGINAL sequence (not the edited one)
#         editedX = X.clone()
        
#         # Extract and insert ORIGINAL CTCFs with flanks
#         if 'orig_CTCFs_coord' in row and row['orig_CTCFs_coord']:
#             orig_ctcfs = row['orig_CTCFs_coord']
            
#             if isinstance(orig_ctcfs, str):
#                 import ast
#                 orig_ctcfs = ast.literal_eval(orig_ctcfs)
            
#             for ctcf_start, ctcf_end, strand in orig_ctcfs:
#                 if 0 <= ctcf_start < self.bin_size and 0 <= ctcf_end <= self.bin_size:
#                     # Extract CTCF with flanks from edited slice
#                     extract_start = max(0, ctcf_start - self.flank_size)
#                     extract_end = min(self.bin_size, ctcf_end + self.flank_size)
                    
#                     ctcf_with_flanks = slice_edited[:, :, extract_start:extract_end]
                    
#                     # Insert into original sequence at the same position
#                     abs_start = edit_start + extract_start
#                     abs_end = edit_start + extract_end
                    
#                     editedX[:, :, abs_start:abs_end] = ctcf_with_flanks
        
#         # Extract and insert NEW CTCFs with flanks
#         if 'new_CTCFs_coord' in row and row['new_CTCFs_coord']:
#             new_ctcfs = row['new_CTCFs_coord']
            
#             if isinstance(new_ctcfs, str):
#                 import ast
#                 new_ctcfs = ast.literal_eval(new_ctcfs)
            
#             for ctcf_start, ctcf_end, strand in new_ctcfs:
#                 if 0 <= ctcf_start < self.bin_size and 0 <= ctcf_end <= self.bin_size:
#                     # Extract CTCF with flanks from edited slice
#                     extract_start = max(0, ctcf_start - self.flank_size)
#                     extract_end = min(self.bin_size, ctcf_end + self.flank_size)
                    
#                     ctcf_with_flanks = slice_edited[:, :, extract_start:extract_end]
                    
#                     # Insert into original sequence at the same position
#                     abs_start = edit_start + extract_start
#                     abs_end = edit_start + extract_end
                    
#                     editedX[:, :, abs_start:abs_end] = ctcf_with_flanks
        
#         # if idx < 3:
#             # orig_count = len(row['orig_CTCFs_coord']) if 'orig_CTCFs_coord' in row and row['orig_CTCFs_coord'] else 0
#             # new_count = len(row['new_CTCFs_coord']) if 'new_CTCFs_coord' in row and row['new_CTCFs_coord'] else 0
#             # print(f"Sequence {idx}: Inserted {orig_count} original + {new_count} new CTCFs (with ±{self.flank_size}bp flanks) into original sequence")
        
#         editedX = editedX.squeeze(0)
        
#         return editedX

In [ ]:
# ### REPLACING G AND C HOMOPOLYMERS WITH AT REPEATS

# import re
# import torch

# class BoundaryGenerationDataset(Dataset):
#     def __init__(self, coord_df, init_seq_path, slice_path, slice=256, cropping=64, bin_size=2048):
#         self.coords = coord_df
#         self.init_seq_path = init_seq_path
#         self.slice_path = slice_path
#         self.slice = slice
#         self.cropping = cropping
#         self.bin_size = bin_size
        
#     def __len__(self):
#         return len(self.coords)
    
#     def sequence_to_tensor(self, sequence, device):
#         """Convert DNA sequence string to one-hot tensor."""
#         mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
#         tensor = torch.zeros(4, len(sequence), device=device)
        
#         for i, base in enumerate(sequence):
#             if base in mapping:
#                 tensor[mapping[base], i] = 1
        
#         return tensor
    
#     def tensor_to_sequence(self, seq_tensor):
#         """Convert one-hot tensor to DNA sequence string."""
#         if seq_tensor.dim() == 3:
#             seq_tensor = seq_tensor.squeeze(0)
        
#         nucleotides = ['A', 'C', 'G', 'T']
#         indices = torch.argmax(seq_tensor, dim=0)
#         sequence = ''.join([nucleotides[i] for i in indices])
#         return sequence
    
#     def replace_gc_homopolymers(self, slice_tensor, min_length=4):
#         """
#         Replace G and C homopolymers (length >= min_length) with AT repeats.
        
#         Args:
#             slice_tensor: [1, 4, 2048] tensor
#             min_length: minimum homopolymer length to replace (default 4)
        
#         Returns:
#             Modified tensor with GC homopolymers replaced
#         """
#         device = slice_tensor.device
        
#         # Convert to sequence
#         sequence = self.tensor_to_sequence(slice_tensor)
        
#         # Find and replace C homopolymers (CCCC+ -> ATAT...)
#         pattern_c = f'C{{{min_length},}}'
#         for match in re.finditer(pattern_c, sequence):
#             length = len(match.group())
#             # Create AT repeat of the same length
#             # replacement = 'AT' * (length // 2) + ('A' if length % 2 == 1 else '')
#             replacement = 'GC' * (length // 2) + ('C' if length % 2 == 1 else '')
#             sequence = sequence[:match.start()] + replacement + sequence[match.end():]
        
#         # Find and replace G homopolymers (GGGG+ -> ATAT...)
#         pattern_g = f'G{{{min_length},}}'
#         for match in re.finditer(pattern_g, sequence):
#             length = len(match.group())
#             # Create AT repeat of the same length
#             # replacement = 'AT' * (length // 2) + ('A' if length % 2 == 1 else '')
#             replacement = 'CG' * (length // 2) + ('C' if length % 2 == 1 else '')
#             sequence = sequence[:match.start()] + replacement + sequence[match.end():]
        
#         # Convert back to tensor
#         modified_tensor = self.sequence_to_tensor(sequence, device)
        
#         return modified_tensor.unsqueeze(0)  # Add batch dimension back

#     def __getitem__(self, idx):
#         row = self.coords.iloc[idx]
#         chrom = row["chrom"]
#         start = row["centered_start"]
#         end = row["centered_end"]
        
#         device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        
#         X = torch.load(f"{self.init_seq_path}{chrom}_{start}_{end}_X.pt", weights_only=True, map_location=device)
#         slice_edited = torch.load(f"{self.slice_path}{chrom}_{start}_{end}_slice.pt", weights_only=True, map_location=device)
        
#         # Replace GC homopolymers (≥4bp) with AT repeats
#         slice_modified = self.replace_gc_homopolymers(slice_edited, min_length=4)
        
#         if idx < 3:
#             # Debug output for first few sequences
#             orig_seq = self.tensor_to_sequence(slice_edited)
#             mod_seq = self.tensor_to_sequence(slice_modified)
            
#             # Count replacements
#             orig_c_runs = len(re.findall(r'C{4,}', orig_seq))
#             orig_g_runs = len(re.findall(r'G{4,}', orig_seq))
#             mod_c_runs = len(re.findall(r'C{4,}', mod_seq))
#             mod_g_runs = len(re.findall(r'G{4,}', mod_seq))
            
#             print(f"\nSequence {idx}:")
#             print(f"  Original: C-runs(≥4bp)={orig_c_runs}, G-runs(≥4bp)={orig_g_runs}")
#             print(f"  Modified: C-runs(≥4bp)={mod_c_runs}, G-runs(≥4bp)={mod_g_runs}")
#             print(f"  Replaced: {orig_c_runs + orig_g_runs - mod_c_runs - mod_g_runs} homopolymers")
        
#         edit_start = (self.slice + self.cropping) * self.bin_size
#         edit_end = edit_start + self.bin_size
        
#         editedX = X.clone()
#         editedX[:, :, edit_start:edit_end] = slice_modified
        
#         editedX = editedX.squeeze(0)
        
#         return editedX

In [ ]:
# ### REPLACING G AND C HOMOPOLYMERS WITH GC REPEATS (PRESERVING GC CONTENT)

# import re
# import torch

# class BoundaryGenerationDataset(Dataset):
#     def __init__(self, coord_df, init_seq_path, slice_path, slice=256, cropping=64, bin_size=2048):
#         self.coords = coord_df
#         self.init_seq_path = init_seq_path
#         self.slice_path = slice_path
#         self.slice = slice
#         self.cropping = cropping
#         self.bin_size = bin_size
        
#     def __len__(self):
#         return len(self.coords)
    
#     def sequence_to_tensor(self, sequence, device):
#         """Convert DNA sequence string to one-hot tensor."""
#         mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
#         tensor = torch.zeros(4, len(sequence), device=device)
        
#         for i, base in enumerate(sequence):
#             if base in mapping:
#                 tensor[mapping[base], i] = 1
        
#         return tensor
    
#     def tensor_to_sequence(self, seq_tensor):
#         """Convert one-hot tensor to DNA sequence string."""
#         if seq_tensor.dim() == 3:
#             seq_tensor = seq_tensor.squeeze(0)
        
#         nucleotides = ['A', 'C', 'G', 'T']
#         indices = torch.argmax(seq_tensor, dim=0)
#         sequence = ''.join([nucleotides[i] for i in indices])
#         return sequence
    
#     def replace_gc_homopolymers_with_gc_repeat(self, slice_tensor, min_length=4):
#         """
#         Replace G and C homopolymers with GC repeats, preserving GC content.
        
#         Strategy:
#         - CCCCC → GCGCG (if odd length, end with C)
#         - GGGGG → CGCGC (if odd length, end with C)
        
#         This maintains:
#         - Exact GC content
#         - Same length
#         - Disrupts homopolymers
        
#         Args:
#             slice_tensor: [1, 4, 2048] tensor
#             min_length: minimum homopolymer length to replace (default 4)
        
#         Returns:
#             Modified tensor with GC homopolymers replaced
#         """
#         device = slice_tensor.device
        
#         # Convert to sequence
#         sequence = self.tensor_to_sequence(slice_tensor)
        
#         # Find and replace C homopolymers (CCCC+ -> GCGC...)
#         pattern_c = f'C{{{min_length},}}'
#         for match in re.finditer(pattern_c, sequence):
#             length = len(match.group())
#             # Create GC repeat: GCGCGC...
#             # If odd length, end with C to maintain exact C count
#             replacement = 'GC' * (length // 2) + ('C' if length % 2 == 1 else '')
#             sequence = sequence[:match.start()] + replacement + sequence[match.end():]
        
#         # Find and replace G homopolymers (GGGG+ -> CGCG...)
#         pattern_g = f'G{{{min_length},}}'
#         for match in re.finditer(pattern_g, sequence):
#             length = len(match.group())
#             # Create CG repeat: CGCGCG...
#             # If odd length, end with C to maintain exact G count
#             # Wait, we need to end with G if odd to maintain G count
#             replacement = 'CG' * (length // 2) + ('G' if length % 2 == 1 else '')
#             sequence = sequence[:match.start()] + replacement + sequence[match.end():]
        
#         # Convert back to tensor
#         modified_tensor = self.sequence_to_tensor(sequence, device)
        
#         return modified_tensor.unsqueeze(0)  # Add batch dimension back

#     def __getitem__(self, idx):
#         row = self.coords.iloc[idx]
#         chrom = row["chrom"]
#         start = row["centered_start"]
#         end = row["centered_end"]
#         fold = row["fold"]
        
#         device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        
#         X = torch.load(f"{self.init_seq_path}/fold{fold}/{chrom}_{start}_{end}_X.pt", weights_only=True, map_location=device)
#         slice_edited = torch.load(f"{self.slice_path}/fold{fold}/{chrom}_{start}_{end}_slice.pt", weights_only=True, map_location=device)
        
#         # Replace GC homopolymers (≥4bp) with GC repeats
#         slice_modified = self.replace_gc_homopolymers_with_gc_repeat(slice_edited, min_length=4)
        
#         if idx < 3:
#             # Debug output for first few sequences
#             orig_seq = self.tensor_to_sequence(slice_edited)
#             mod_seq = self.tensor_to_sequence(slice_modified)
            
#             # Count homopolymers
#             orig_c_runs = len(re.findall(r'C{4,}', orig_seq))
#             orig_g_runs = len(re.findall(r'G{4,}', orig_seq))
#             mod_c_runs = len(re.findall(r'C{4,}', mod_seq))
#             mod_g_runs = len(re.findall(r'G{4,}', mod_seq))
            
#             # Count GC repeats
#             orig_gcgc = len(re.findall(r'GCGC', orig_seq))
#             orig_cgcg = len(re.findall(r'CGCG', orig_seq))
#             mod_gcgc = len(re.findall(r'GCGC', mod_seq))
#             mod_cgcg = len(re.findall(r'CGCG', mod_seq))
            
#             # Verify GC content
#             orig_gc = (orig_seq.count('G') + orig_seq.count('C')) / len(orig_seq)
#             mod_gc = (mod_seq.count('G') + mod_seq.count('C')) / len(mod_seq)
            
#             print(f"\nSequence {idx}:")
#             print(f"  Homopolymers (≥4bp):")
#             print(f"    C-runs: {orig_c_runs} → {mod_c_runs} ({mod_c_runs - orig_c_runs:+d})")
#             print(f"    G-runs: {orig_g_runs} → {mod_g_runs} ({mod_g_runs - orig_g_runs:+d})")
#             print(f"    Total replaced: {orig_c_runs + orig_g_runs - mod_c_runs - mod_g_runs}")
            
#             print(f"  GC repeats:")
#             print(f"    GCGC: {orig_gcgc} → {mod_gcgc} ({mod_gcgc - orig_gcgc:+d})")
#             print(f"    CGCG: {orig_cgcg} → {mod_cgcg} ({mod_cgcg - orig_cgcg:+d})")
            
#             print(f"  GC content:")
#             print(f"    Original: {orig_gc:.4f}")
#             print(f"    Modified: {mod_gc:.4f}")
#             print(f"    Difference: {mod_gc - orig_gc:.6f} (should be ~0)")
            
#             # Show an example replacement
#             for match in re.finditer(r'C{4,}', orig_seq):
#                 print(f"  Example C-homopolymer replacement:")
#                 print(f"    Original: {orig_seq[match.start():match.end()]}")
#                 print(f"    Modified: {mod_seq[match.start():match.end()]}")
#                 break
            
#             for match in re.finditer(r'G{4,}', orig_seq):
#                 print(f"  Example G-homopolymer replacement:")
#                 print(f"    Original: {orig_seq[match.start():match.end()]}")
#                 print(f"    Modified: {mod_seq[match.start():match.end()]}")
#                 break
        
#         edit_start = (self.slice + self.cropping) * self.bin_size
#         edit_end = edit_start + self.bin_size
        
#         editedX = X.clone()
#         editedX[:, :, edit_start:edit_end] = slice_modified
        
#         editedX = editedX.squeeze(0)
        
#         return editedX

In [ ]:
# import re
# import torch

# class BoundaryGenerationDataset(Dataset):
#     def __init__(self, coord_df, init_seq_path, slice_path, slice=256, cropping=64, bin_size=2048):
#         self.coords = coord_df
#         self.init_seq_path = init_seq_path
#         self.slice_path = slice_path
#         self.slice = slice
#         self.cropping = cropping
#         self.bin_size = bin_size
        
#     def __len__(self):
#         return len(self.coords)
    
#     def sequence_to_tensor(self, sequence, device):
#         """Convert DNA sequence string to one-hot tensor."""
#         mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
#         tensor = torch.zeros(4, len(sequence), device=device)
        
#         for i, base in enumerate(sequence):
#             if base in mapping:
#                 tensor[mapping[base], i] = 1
        
#         return tensor
    
#     def tensor_to_sequence(self, seq_tensor):
#         """Convert one-hot tensor to DNA sequence string."""
#         if seq_tensor.dim() == 3:
#             seq_tensor = seq_tensor.squeeze(0)
        
#         nucleotides = ['A', 'C', 'G', 'T']
#         indices = torch.argmax(seq_tensor, dim=0)
#         sequence = ''.join([nucleotides[i] for i in indices])
#         return sequence
    
#     def replace_at_homopolymers_with_at_repeat(self, slice_tensor, min_length=4):
#         """
#         Replace A and T homopolymers with AT repeats, preserving AT content.
        
#         Strategy:
#         - AAAAA → TATAT (if odd length, end with A)
#         - TTTTT → ATATAT (if odd length, end with T)
        
#         This maintains:
#         - Exact AT content
#         - Same length
#         - Disrupts homopolymers
        
#         Args:
#             slice_tensor: [1, 4, 2048] tensor
#             min_length: minimum homopolymer length to replace (default 4)
        
#         Returns:
#             Modified tensor with AT homopolymers replaced
#         """
#         device = slice_tensor.device
        
#         # Convert to sequence
#         sequence = self.tensor_to_sequence(slice_tensor)
        
#         # Find and replace A homopolymers (AAAA+ -> TATA...)
#         pattern_a = f'A{{{min_length},}}'
#         for match in re.finditer(pattern_a, sequence):
#             length = len(match.group())
#             # Create TA repeat: TATATA...
#             # If odd length, end with A to maintain exact A count
#             replacement = 'TA' * (length // 2) + ('A' if length % 2 == 1 else '')
#             sequence = sequence[:match.start()] + replacement + sequence[match.end():]
        
#         # Find and replace T homopolymers (TTTT+ -> ATAT...)
#         pattern_t = f'T{{{min_length},}}'
#         for match in re.finditer(pattern_t, sequence):
#             length = len(match.group())
#             # Create AT repeat: ATATAT...
#             # If odd length, end with T to maintain exact T count
#             replacement = 'AT' * (length // 2) + ('T' if length % 2 == 1 else '')
#             sequence = sequence[:match.start()] + replacement + sequence[match.end():]
        
#         # Convert back to tensor
#         modified_tensor = self.sequence_to_tensor(sequence, device)
        
#         return modified_tensor.unsqueeze(0)  # Add batch dimension back

#     def __getitem__(self, idx):
#         row = self.coords.iloc[idx]
#         chrom = row["chrom"]
#         start = row["centered_start"]
#         end = row["centered_end"]
        
#         device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        
#         X = torch.load(f"{self.init_seq_path}{chrom}_{start}_{end}_X.pt", weights_only=True, map_location=device)
#         slice_edited = torch.load(f"{self.slice_path}{chrom}_{start}_{end}_slice.pt", weights_only=True, map_location=device)
        
#         # Replace AT homopolymers (≥4bp) with AT repeats
#         slice_modified = self.replace_at_homopolymers_with_at_repeat(slice_edited, min_length=4)
        
#         if idx < 3:
#             # Debug output for first few sequences
#             orig_seq = self.tensor_to_sequence(slice_edited)
#             mod_seq = self.tensor_to_sequence(slice_modified)
            
#             # Count homopolymers
#             orig_a_runs = len(re.findall(r'A{4,}', orig_seq))
#             orig_t_runs = len(re.findall(r'T{4,}', orig_seq))
#             mod_a_runs = len(re.findall(r'A{4,}', mod_seq))
#             mod_t_runs = len(re.findall(r'T{4,}', mod_seq))
            
#             # Count AT repeats
#             orig_tata = len(re.findall(r'TATA', orig_seq))
#             orig_atat = len(re.findall(r'ATAT', orig_seq))
#             mod_tata = len(re.findall(r'TATA', mod_seq))
#             mod_atat = len(re.findall(r'ATAT', mod_seq))
            
#             # Verify AT content
#             orig_at = (orig_seq.count('A') + orig_seq.count('T')) / len(orig_seq)
#             mod_at = (mod_seq.count('A') + mod_seq.count('T')) / len(mod_seq)
            
#             print(f"\nSequence {idx}:")
#             print(f"  Homopolymers (≥4bp):")
#             print(f"    A-runs: {orig_a_runs} → {mod_a_runs} ({mod_a_runs - orig_a_runs:+d})")
#             print(f"    T-runs: {orig_t_runs} → {mod_t_runs} ({mod_t_runs - orig_t_runs:+d})")
#             print(f"    Total replaced: {orig_a_runs + orig_t_runs - mod_a_runs - mod_t_runs}")
            
#             print(f"  AT repeats:")
#             print(f"    TATA: {orig_tata} → {mod_tata} ({mod_tata - orig_tata:+d})")
#             print(f"    ATAT: {orig_atat} → {mod_atat} ({mod_atat - orig_atat:+d})")
            
#             print(f"  AT content:")
#             print(f"    Original: {orig_at:.4f}")
#             print(f"    Modified: {mod_at:.4f}")
#             print(f"    Difference: {mod_at - orig_at:.6f} (should be ~0)")
            
#             # Show an example replacement
#             for match in re.finditer(r'A{4,}', orig_seq):
#                 print(f"  Example A-homopolymer replacement:")
#                 print(f"    Original: {orig_seq[match.start():match.end()]}")
#                 print(f"    Modified: {mod_seq[match.start():match.end()]}")
#                 break
            
#             for match in re.finditer(r'T{4,}', orig_seq):
#                 print(f"  Example T-homopolymer replacement:")
#                 print(f"    Original: {orig_seq[match.start():match.end()]}")
#                 print(f"    Modified: {mod_seq[match.start():match.end()]}")
#                 break
        
#         edit_start = (self.slice + self.cropping) * self.bin_size
#         edit_end = edit_start + self.bin_size
        
#         editedX = X.clone()
#         editedX[:, :, edit_start:edit_end] = slice_modified
        
#         editedX = editedX.squeeze(0)
        
#         return editedX

In [ ]:
# ### REPLACING G AND C HOMOPOLYMERS WITH GC REPEATS - CONFIGURABLE MIN LENGTH

# import re
# import torch

# class BoundaryGenerationDataset(Dataset):
#     def __init__(self, coord_df, init_seq_path, slice_path, slice=256, cropping=64, bin_size=2048, min_homopolymer_length=4):
#         """
#         Args:
#             coord_df: DataFrame with sequence coordinates
#             init_seq_path: Path to initial sequences
#             slice_path: Path to edited slices
#             slice: Slice parameter
#             cropping: Cropping parameter
#             bin_size: Bin size
#             min_homopolymer_length: Minimum length of homopolymers to replace (default: 4)
#                                    - Use 3 to replace GGG, CCC and longer
#                                    - Use 4 to replace GGGG, CCCC and longer (typical)
#                                    - Use 5 to replace GGGGG, CCCCC and longer
#                                    - Use 6 to replace GGGGGG, CCCCCC and longer
#         """
#         self.coords = coord_df
#         self.init_seq_path = init_seq_path
#         self.slice_path = slice_path
#         self.slice = slice
#         self.cropping = cropping
#         self.bin_size = bin_size
#         self.min_homopolymer_length = min_homopolymer_length
        
#     def __len__(self):
#         return len(self.coords)
    
#     def sequence_to_tensor(self, sequence, device):
#         """Convert DNA sequence string to one-hot tensor."""
#         mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
#         tensor = torch.zeros(4, len(sequence), device=device)
        
#         for i, base in enumerate(sequence):
#             if base in mapping:
#                 tensor[mapping[base], i] = 1
        
#         return tensor
    
#     def tensor_to_sequence(self, seq_tensor):
#         """Convert one-hot tensor to DNA sequence string."""
#         if seq_tensor.dim() == 3:
#             seq_tensor = seq_tensor.squeeze(0)
        
#         nucleotides = ['A', 'C', 'G', 'T']
#         indices = torch.argmax(seq_tensor, dim=0)
#         sequence = ''.join([nucleotides[i] for i in indices])
#         return sequence
    
#     def replace_gc_homopolymers_with_gc_repeat(self, slice_tensor, min_length=None):
#         """
#         Replace G and C homopolymers (≥ min_length) with GC repeats.
        
#         Strategy:
#         - CCCCC → GCGCG (if odd length, end with C)
#         - GGGGG → CGCGC (if odd length, end with G)
        
#         This maintains:
#         - Exact GC content
#         - Same length
#         - Disrupts homopolymers
        
#         Args:
#             slice_tensor: [1, 4, 2048] tensor
#             min_length: minimum homopolymer length to replace
#                        If None, uses self.min_homopolymer_length
        
#         Returns:
#             Modified tensor with GC homopolymers replaced
#         """
#         device = slice_tensor.device
        
#         # Use instance min_length if not specified
#         if min_length is None:
#             min_length = self.min_homopolymer_length
        
#         # Convert to sequence
#         sequence = self.tensor_to_sequence(slice_tensor)
        
#         # Find and replace C homopolymers (CCCC+ -> GCGC...)
#         pattern_c = f'C{{{min_length},}}'
#         for match in re.finditer(pattern_c, sequence):
#             length = len(match.group())
#             # Create GC repeat: GCGCGC...
#             # If odd length, end with C to maintain exact C count
#             replacement = 'GC' * (length // 2) + ('C' if length % 2 == 1 else '')
#             sequence = sequence[:match.start()] + replacement + sequence[match.end():]
        
#         # Find and replace G homopolymers (GGGG+ -> CGCG...)
#         pattern_g = f'G{{{min_length},}}'
#         for match in re.finditer(pattern_g, sequence):
#             length = len(match.group())
#             # Create CG repeat: CGCGCG...
#             # If odd length, end with G to maintain exact G count
#             replacement = 'CG' * (length // 2) + ('G' if length % 2 == 1 else '')
#             sequence = sequence[:match.start()] + replacement + sequence[match.end():]
        
#         # Convert back to tensor
#         modified_tensor = self.sequence_to_tensor(sequence, device)
        
#         return modified_tensor.unsqueeze(0)  # Add batch dimension back
    
#     def count_homopolymers_by_length(self, sequence):
#         """Count homopolymers of different lengths."""
#         counts = {}
#         for length in range(3, 11):  # 3bp to 10bp
#             counts[f'G{length}+'] = len(re.findall(f'G{{{length},}}', sequence))
#             counts[f'C{length}+'] = len(re.findall(f'C{{{length},}}', sequence))
#         return counts

#     def __getitem__(self, idx):
#         row = self.coords.iloc[idx]
#         chrom = row["chrom"]
#         start = row["centered_start"]
#         end = row["centered_end"]
        
#         device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        
#         X = torch.load(f"{self.init_seq_path}{chrom}_{start}_{end}_X.pt", weights_only=True, map_location=device)
#         slice_edited = torch.load(f"{self.slice_path}{chrom}_{start}_{end}_slice.pt", weights_only=True, map_location=device)
        
#         # Replace GC homopolymers with configured min_length
#         slice_modified = self.replace_gc_homopolymers_with_gc_repeat(slice_edited)
        
#         if idx < 3:
#             # Debug output for first few sequences
#             orig_seq = self.tensor_to_sequence(slice_edited)
#             mod_seq = self.tensor_to_sequence(slice_modified)
            
#             # Count homopolymers by length
#             orig_counts = self.count_homopolymers_by_length(orig_seq)
#             mod_counts = self.count_homopolymers_by_length(mod_seq)
            
#             print(f"\nSequence {idx} (min_length={self.min_homopolymer_length}):")
#             print(f"  Homopolymers by length (before → after):")
            
#             # Show counts for relevant lengths
#             for length in range(3, 9):
#                 g_orig = orig_counts[f'G{length}+']
#                 g_mod = mod_counts[f'G{length}+']
#                 c_orig = orig_counts[f'C{length}+']
#                 c_mod = mod_counts[f'C{length}+']
                
#                 if g_orig > 0 or c_orig > 0 or g_mod > 0 or c_mod > 0:
#                     print(f"    {length}bp: G: {g_orig:2d}→{g_mod:2d}  C: {c_orig:2d}→{c_mod:2d}", end="")
#                     if length >= self.min_homopolymer_length:
#                         print("  [REPLACED]")
#                     else:
#                         print("  [kept]")
            
#             # Count GC repeats
#             orig_gcgc = len(re.findall(r'GCGC', orig_seq))
#             orig_cgcg = len(re.findall(r'CGCG', orig_seq))
#             mod_gcgc = len(re.findall(r'GCGC', mod_seq))
#             mod_cgcg = len(re.findall(r'CGCG', mod_seq))
            
#             print(f"  GC repeats:")
#             print(f"    GCGC: {orig_gcgc:2d} → {mod_gcgc:2d} ({mod_gcgc - orig_gcgc:+d})")
#             print(f"    CGCG: {orig_cgcg:2d} → {mod_cgcg:2d} ({mod_cgcg - orig_cgcg:+d})")
            
#             # Verify GC content
#             orig_gc = (orig_seq.count('G') + orig_seq.count('C')) / len(orig_seq)
#             mod_gc = (mod_seq.count('G') + mod_seq.count('C')) / len(mod_seq)
            
#             print(f"  GC content: {orig_gc:.4f} → {mod_gc:.4f} (diff: {mod_gc - orig_gc:.6f})")
            
#             # Show example replacement
#             pattern = f'[GC]{{{self.min_homopolymer_length},}}'
#             for match in re.finditer(pattern, orig_seq):
#                 print(f"  Example replacement:")
#                 print(f"    Original: {orig_seq[match.start():match.end()]}")
#                 print(f"    Modified: {mod_seq[match.start():match.end()]}")
#                 break
        
#         edit_start = (self.slice + self.cropping) * self.bin_size
#         edit_end = edit_start + self.bin_size
        
#         editedX = X.clone()
#         editedX[:, :, edit_start:edit_end] = slice_modified
        
#         editedX = editedX.squeeze(0)
        
#         return editedX

In [ ]:
# ### REPLACING 50% OF G AND C HOMOPOLYMERS WITH GC REPEATS (DOSE-RESPONSE TEST)

# import re
# import torch
# import numpy as np

# class BoundaryGenerationDataset(Dataset):
#     def __init__(self, coord_df, init_seq_path, slice_path, slice=256, cropping=64, bin_size=2048):
#         self.coords = coord_df
#         self.init_seq_path = init_seq_path
#         self.slice_path = slice_path
#         self.slice = slice
#         self.cropping = cropping
#         self.bin_size = bin_size
        
#     def __len__(self):
#         return len(self.coords)
    
#     def sequence_to_tensor(self, sequence, device):
#         """Convert DNA sequence string to one-hot tensor."""
#         mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
#         tensor = torch.zeros(4, len(sequence), device=device)
        
#         for i, base in enumerate(sequence):
#             if base in mapping:
#                 tensor[mapping[base], i] = 1
        
#         return tensor
    
#     def tensor_to_sequence(self, seq_tensor):
#         """Convert one-hot tensor to DNA sequence string."""
#         if seq_tensor.dim() == 3:
#             seq_tensor = seq_tensor.squeeze(0)
        
#         nucleotides = ['A', 'C', 'G', 'T']
#         indices = torch.argmax(seq_tensor, dim=0)
#         sequence = ''.join([nucleotides[i] for i in indices])
#         return sequence
    
#     def replace_gc_homopolymers_partial(self, slice_tensor, min_length=4, fraction=0.5, seed=None):
#         """
#         Replace only a FRACTION of G and C homopolymers with GC repeats.
        
#         This allows testing dose-response: does disrupting 50% of homopolymers
#         restore 25% of boundary strength? (50% of the 50% effect)
        
#         Strategy:
#         1. Find all homopolymers ≥ min_length
#         2. Randomly select fraction of them (e.g., 50%)
#         3. Replace selected ones: CCCCC → GCGCG, GGGGG → CGCGC
#         4. Keep non-selected ones intact
        
#         Args:
#             slice_tensor: [1, 4, 2048] tensor
#             min_length: minimum homopolymer length to replace (default 4)
#             fraction: fraction of homopolymers to replace (0.0-1.0, default 0.5)
#             seed: random seed for reproducibility
        
#         Returns:
#             Modified tensor with partial homopolymer replacement
#         """
#         device = slice_tensor.device
        
#         # Convert to sequence
#         sequence = self.tensor_to_sequence(slice_tensor)
        
#         # Set random seed for reproducibility
#         if seed is not None:
#             np.random.seed(seed)
        
#         # Find all C homopolymers
#         c_homopolymers = []
#         pattern_c = f'C{{{min_length},}}'
#         for match in re.finditer(pattern_c, sequence):
#             c_homopolymers.append((match.start(), match.end(), 'C'))
        
#         # Find all G homopolymers
#         g_homopolymers = []
#         pattern_g = f'G{{{min_length},}}'
#         for match in re.finditer(pattern_g, sequence):
#             g_homopolymers.append((match.start(), match.end(), 'G'))
        
#         # Combine all homopolymers
#         all_homopolymers = c_homopolymers + g_homopolymers
#         total_count = len(all_homopolymers)
        
#         if total_count == 0:
#             # No homopolymers found
#             return slice_tensor
        
#         # Randomly select fraction of them
#         n_to_replace = int(total_count * fraction)
#         indices_to_replace = np.random.choice(total_count, size=n_to_replace, replace=False)
#         selected_homopolymers = [all_homopolymers[i] for i in indices_to_replace]
        
#         # Sort by position (descending) to replace from end to start
#         # This prevents position shifts during replacement
#         selected_homopolymers.sort(key=lambda x: x[0], reverse=True)
        
#         # Replace selected homopolymers
#         seq_list = list(sequence)
#         for start, end, base_type in selected_homopolymers:
#             length = end - start
            
#             if base_type == 'C':
#                 # CCCCC → GCGCG
#                 replacement = 'GC' * (length // 2) + ('C' if length % 2 == 1 else '')
#             else:  # base_type == 'G'
#                 # GGGGG → CGCGC
#                 replacement = 'CG' * (length // 2) + ('G' if length % 2 == 1 else '')
            
#             # Replace in the list
#             seq_list[start:end] = list(replacement)
        
#         # Convert back to sequence and tensor
#         modified_sequence = ''.join(seq_list)
#         modified_tensor = self.sequence_to_tensor(modified_sequence, device)
        
#         return modified_tensor.unsqueeze(0)  # Add batch dimension back

#     def __getitem__(self, idx):
#         row = self.coords.iloc[idx]
#         chrom = row["chrom"]
#         start = row["centered_start"]
#         end = row["centered_end"]
        
#         device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        
#         X = torch.load(f"{self.init_seq_path}{chrom}_{start}_{end}_X.pt", weights_only=True, map_location=device)
#         slice_edited = torch.load(f"{self.slice_path}{chrom}_{start}_{end}_slice.pt", weights_only=True, map_location=device)
        
#         # Replace 50% of GC homopolymers (≥4bp) with GC repeats
#         slice_modified = self.replace_gc_homopolymers_partial(
#             slice_edited, 
#             min_length=4, 
#             fraction=0.5,  # Replace 50%
#             seed=idx  # Use idx as seed for reproducibility
#         )
        
#         if idx < 3:
#             # Debug output for first few sequences
#             orig_seq = self.tensor_to_sequence(slice_edited)
#             mod_seq = self.tensor_to_sequence(slice_modified)
            
#             # Count homopolymers
#             orig_c_runs = len(re.findall(r'C{4,}', orig_seq))
#             orig_g_runs = len(re.findall(r'G{4,}', orig_seq))
#             mod_c_runs = len(re.findall(r'C{4,}', mod_seq))
#             mod_g_runs = len(re.findall(r'G{4,}', mod_seq))
            
#             total_orig = orig_c_runs + orig_g_runs
#             total_mod = mod_c_runs + mod_g_runs
#             replaced = total_orig - total_mod
            
#             # Count GC repeats
#             orig_gcgc = len(re.findall(r'GCGC', orig_seq))
#             orig_cgcg = len(re.findall(r'CGCG', orig_seq))
#             mod_gcgc = len(re.findall(r'GCGC', mod_seq))
#             mod_cgcg = len(re.findall(r'CGCG', mod_seq))
            
#             # Verify GC content
#             orig_gc = (orig_seq.count('G') + orig_seq.count('C')) / len(orig_seq)
#             mod_gc = (mod_seq.count('G') + mod_seq.count('C')) / len(mod_seq)
            
#             print(f"\nSequence {idx}:")
#             print(f"  Homopolymers (≥4bp):")
#             print(f"    Total found: {total_orig}")
#             print(f"    Replaced: {replaced} (~{100*replaced/total_orig if total_orig > 0 else 0:.1f}%, target: 50%)")
#             print(f"    Remaining: {total_mod}")
#             print(f"    C-runs: {orig_c_runs} → {mod_c_runs} ({mod_c_runs - orig_c_runs:+d})")
#             print(f"    G-runs: {orig_g_runs} → {mod_g_runs} ({mod_g_runs - orig_g_runs:+d})")
            
#             print(f"  GC repeats (increased due to replacement):")
#             print(f"    GCGC: {orig_gcgc} → {mod_gcgc} ({mod_gcgc - orig_gcgc:+d})")
#             print(f"    CGCG: {orig_cgcg} → {mod_cgcg} ({mod_cgcg - orig_cgcg:+d})")
            
#             print(f"  GC content:")
#             print(f"    Original: {orig_gc:.4f}")
#             print(f"    Modified: {mod_gc:.4f}")
#             print(f"    Difference: {mod_gc - orig_gc:.6f} (should be ~0)")
        
#         edit_start = (self.slice + self.cropping) * self.bin_size
#         edit_end = edit_start + self.bin_size
        
#         editedX = X.clone()
#         editedX[:, :, edit_start:edit_end] = slice_modified
        
#         editedX = editedX.squeeze(0)
        
#         return editedX

In [ ]:
# ### DISRUPTING GC/CG REPEATS WHILE PRESERVING G AND C HOMOPOLYMERS

# import re
# import torch
# import numpy as np

# class BoundaryGenerationDataset(Dataset):
#     def __init__(self, coord_df, init_seq_path, slice_path, slice=256, cropping=64, bin_size=2048):
#         self.coords = coord_df
#         self.init_seq_path = init_seq_path
#         self.slice_path = slice_path
#         self.slice = slice
#         self.cropping = cropping
#         self.bin_size = bin_size
        
#     def __len__(self):
#         return len(self.coords)
    
#     def sequence_to_tensor(self, sequence, device):
#         """Convert DNA sequence string to one-hot tensor."""
#         mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
#         tensor = torch.zeros(4, len(sequence), device=device)
        
#         for i, base in enumerate(sequence):
#             if base in mapping:
#                 tensor[mapping[base], i] = 1
        
#         return tensor
    
#     def tensor_to_sequence(self, seq_tensor):
#         """Convert one-hot tensor to DNA sequence string."""
#         if seq_tensor.dim() == 3:
#             seq_tensor = seq_tensor.squeeze(0)
        
#         nucleotides = ['A', 'C', 'G', 'T']
#         indices = torch.argmax(seq_tensor, dim=0)
#         sequence = ''.join([nucleotides[i] for i in indices])
#         return sequence
    
#     def identify_homopolymers(self, sequence, min_length=4):
#         """
#         Find positions of G and C homopolymers.
        
#         Args:
#             sequence: DNA sequence string
#             min_length: minimum homopolymer length to protect
        
#         Returns:
#             Set of positions (indices) that are in homopolymers
#         """
#         protected_positions = set()
        
#         # Find G homopolymers (GGGG+)
#         for match in re.finditer(f'G{{{min_length},}}', sequence):
#             for pos in range(match.start(), match.end()):
#                 protected_positions.add(pos)
        
#         # Find C homopolymers (CCCC+)
#         for match in re.finditer(f'C{{{min_length},}}', sequence):
#             for pos in range(match.start(), match.end()):
#                 protected_positions.add(pos)
        
#         return protected_positions
    
#     def disrupt_gc_repeats_preserve_homopolymers(self, slice_tensor, min_homopolymer_length=4, min_repeat_length=2, seed=None):
#         """
#         Disrupt GC/CG dinucleotide repeats while preserving everything else.
        
#         Strategy:
#         1. Identify G and C homopolymers (≥4bp) - PROTECT
#         2. Identify GC/CG repeats (≥2 dinucleotides = GCGC, CGCG, etc.) - SHUFFLE
#         3. Leave everything else unchanged
        
#         Example:
#             ATGGGGCGCGCGCCCCCTA
#             --GGGG-------CCCCC--  ← Protected (homopolymers)
#             ------CGCGCG--------  ← Shuffled (GC repeat)
#             AT----      ------TA  ← Unchanged
        
#         Args:
#             slice_tensor: [1, 4, 2048] tensor
#             min_homopolymer_length: minimum length to protect (default 4)
#             min_repeat_length: minimum number of GC/CG repeats to disrupt (default 2 = GCGC or CGCG)
#             seed: random seed for reproducibility
        
#         Returns:
#             Modified tensor with only GC repeats disrupted
#         """
#         device = slice_tensor.device
        
#         # Convert to sequence
#         sequence = self.tensor_to_sequence(slice_tensor)
#         seq_list = list(sequence)
        
#         # Step 1: Identify protected positions (homopolymers)
#         protected_positions = self.identify_homopolymers(sequence, min_homopolymer_length)
        
#         # Step 2: Find GC/CG repeats (not in protected regions)
#         # Pattern: (GC){2,} or (CG){2,}
#         # This matches GCGC, GCGCGC, CGCG, CGCGCG, etc.
#         gc_repeat_pattern = f'(?:GC){{{min_repeat_length},}}'
#         cg_repeat_pattern = f'(?:CG){{{min_repeat_length},}}'
        
#         repeat_regions = []
        
#         # Find GC repeats
#         for match in re.finditer(gc_repeat_pattern, sequence):
#             # Check if this repeat overlaps with any homopolymer
#             overlap = False
#             for pos in range(match.start(), match.end()):
#                 if pos in protected_positions:
#                     overlap = True
#                     break
            
#             if not overlap:
#                 repeat_regions.append((match.start(), match.end()))
        
#         # Find CG repeats
#         for match in re.finditer(cg_repeat_pattern, sequence):
#             # Check if this repeat overlaps with any homopolymer
#             overlap = False
#             for pos in range(match.start(), match.end()):
#                 if pos in protected_positions:
#                     overlap = True
#                     break
            
#             if not overlap:
#                 # Check if already in repeat_regions (avoid duplicates)
#                 already_added = False
#                 for start, end in repeat_regions:
#                     if not (match.end() <= start or match.start() >= end):
#                         already_added = True
#                         break
                
#                 if not already_added:
#                     repeat_regions.append((match.start(), match.end()))
        
#         # Step 3: Shuffle only the repeat regions
#         if seed is not None:
#             np.random.seed(seed)
        
#         for start, end in repeat_regions:
#             if end - start > 1:
#                 region_bases = seq_list[start:end]
#                 np.random.shuffle(region_bases)
#                 seq_list[start:end] = region_bases
        
#         # Convert back to sequence and tensor
#         modified_sequence = ''.join(seq_list)
#         modified_tensor = self.sequence_to_tensor(modified_sequence, device)
        
#         return modified_tensor.unsqueeze(0)  # Add batch dimension back
    
#     def count_patterns(self, sequence):
#         """Count various sequence patterns for debugging."""
#         patterns = {
#             'G4+': len(re.findall(r'G{4,}', sequence)),
#             'C4+': len(re.findall(r'C{4,}', sequence)),
#             'G5+': len(re.findall(r'G{5,}', sequence)),
#             'C5+': len(re.findall(r'C{5,}', sequence)),
#             'GCGC': len(re.findall(r'GCGC', sequence)),
#             'CGCG': len(re.findall(r'CGCG', sequence)),
#             'GC_dinuc': sequence.count('GC'),
#             'CG_dinuc': sequence.count('CG'),
#         }
#         return patterns

#     def __getitem__(self, idx):
#         row = self.coords.iloc[idx]
#         chrom = row["chrom"]
#         start = row["centered_start"]
#         end = row["centered_end"]
        
#         device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        
#         X = torch.load(f"{self.init_seq_path}{chrom}_{start}_{end}_X.pt", weights_only=True, map_location=device)
#         slice_edited = torch.load(f"{self.slice_path}{chrom}_{start}_{end}_slice.pt", weights_only=True, map_location=device)
        
#         # Disrupt GC repeats while preserving homopolymers
#         slice_modified = self.disrupt_gc_repeats_preserve_homopolymers(
#             slice_edited, 
#             min_homopolymer_length=4,
#             seed=idx  # Use idx as seed for reproducibility
#         )
        
#         if idx < 3:
#             # Debug output for first few sequences
#             orig_seq = self.tensor_to_sequence(slice_edited)
#             mod_seq = self.tensor_to_sequence(slice_modified)
            
#             orig_patterns = self.count_patterns(orig_seq)
#             mod_patterns = self.count_patterns(mod_seq)
            
#             print(f"\nSequence {idx}:")
#             print(f"  Homopolymers (should be UNCHANGED):")
#             print(f"    G4+: {orig_patterns['G4+']:3d} → {mod_patterns['G4+']:3d} ({mod_patterns['G4+'] - orig_patterns['G4+']:+2d})")
#             print(f"    C4+: {orig_patterns['C4+']:3d} → {mod_patterns['C4+']:3d} ({mod_patterns['C4+'] - orig_patterns['C4+']:+2d})")
#             print(f"    G5+: {orig_patterns['G5+']:3d} → {mod_patterns['G5+']:3d} ({mod_patterns['G5+'] - orig_patterns['G5+']:+2d})")
#             print(f"    C5+: {orig_patterns['C5+']:3d} → {mod_patterns['C5+']:3d} ({mod_patterns['C5+'] - orig_patterns['C5+']:+2d})")
            
#             print(f"  GC repeats (should be DISRUPTED):")
#             print(f"    GCGC: {orig_patterns['GCGC']:3d} → {mod_patterns['GCGC']:3d} ({mod_patterns['GCGC'] - orig_patterns['GCGC']:+3d})")
#             print(f"    CGCG: {orig_patterns['CGCG']:3d} → {mod_patterns['CGCG']:3d} ({mod_patterns['CGCG'] - orig_patterns['CGCG']:+3d})")
            
#             print(f"  Dinucleotides (may change slightly):")
#             print(f"    GC: {orig_patterns['GC_dinuc']:3d} → {mod_patterns['GC_dinuc']:3d} ({mod_patterns['GC_dinuc'] - orig_patterns['GC_dinuc']:+3d})")
#             print(f"    CG: {orig_patterns['CG_dinuc']:3d} → {mod_patterns['CG_dinuc']:3d} ({mod_patterns['CG_dinuc'] - orig_patterns['CG_dinuc']:+3d})")
            
#             # Verify GC content unchanged
#             orig_gc = (orig_seq.count('G') + orig_seq.count('C')) / len(orig_seq)
#             mod_gc = (mod_seq.count('G') + mod_seq.count('C')) / len(mod_seq)
#             print(f"  GC content: {orig_gc:.4f} → {mod_gc:.4f} (diff: {mod_gc - orig_gc:.6f})")
            
#             # Show example of a disrupted repeat
#             for match in re.finditer(r'(?:GC){2,}|(?:CG){2,}', orig_seq):
#                 if match.end() - match.start() >= 4:  # At least GCGC
#                     print(f"  Example disrupted repeat:")
#                     print(f"    Original: {orig_seq[match.start():match.end()]}")
#                     print(f"    Modified: {mod_seq[match.start():match.end()]}")
#                     break
        
#         edit_start = (self.slice + self.cropping) * self.bin_size
#         edit_end = edit_start + self.bin_size
        
#         editedX = X.clone()
#         editedX[:, :, edit_start:edit_start + self.bin_size] = slice_modified
        
#         editedX = editedX.squeeze(0)
        
#         return editedX

In [ ]:
# ### SHUFFLING UP00088_1 (Plagl1) MOTIF INSTANCES

# import re
# import torch
# import numpy as np

# class BoundaryGenerationDataset(Dataset):
#     def __init__(self, coord_df, init_seq_path, slice_path, slice=256, cropping=64, bin_size=2048):
#         self.coords = coord_df
#         self.init_seq_path = init_seq_path
#         self.slice_path = slice_path
#         self.slice = slice
#         self.cropping = cropping
#         self.bin_size = bin_size
        
#     def __len__(self):
#         return len(self.coords)
    
#     def sequence_to_tensor(self, sequence, device):
#         """Convert DNA sequence string to one-hot tensor."""
#         mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
#         tensor = torch.zeros(4, len(sequence), device=device)
        
#         for i, base in enumerate(sequence):
#             if base in mapping:
#                 tensor[mapping[base], i] = 1
        
#         return tensor
    
#     def tensor_to_sequence(self, seq_tensor):
#         """Convert one-hot tensor to DNA sequence string."""
#         if seq_tensor.dim() == 3:
#             seq_tensor = seq_tensor.squeeze(0)
        
#         nucleotides = ['A', 'C', 'G', 'T']
#         indices = torch.argmax(seq_tensor, dim=0)
#         sequence = ''.join([nucleotides[i] for i in indices])
#         return sequence
    
#     def find_plagl1_motifs(self, sequence):
#         """
#         Find instances of UP00088_1 (Plagl1) motif core.
        
#         Based on TOMTOM results:
#         - Query consensus: AGGGCGCCCCCGAGA (Filter 6)
#         - Target consensus: CTAGGGGCGCCCCCAA (UP00088_1)
#         - Core overlap: GGGCGCCC (8bp core)
        
#         Using shorter core (GGGCGCCC) to capture more instances
#         and be less restrictive than the full homopolymer.
        
#         Returns:
#             List of (start, end) tuples for motif instances
#         """
#         motif_instances = []
        
#         # Shorter core pattern - more sensitive
#         core_pattern = r'GGGCGCCC'  # 8bp core (GGG + CGCCC)
        
#         # Find all instances
#         for match in re.finditer(core_pattern, sequence):
#             motif_instances.append((match.start(), match.end()))
        
#         return motif_instances
    
#     def shuffle_plagl1_motifs(self, slice_tensor, seed=None):
#         """
#         Find and shuffle UP00088_1 (Plagl1) motif instances.
        
#         Strategy:
#         1. Find all instances of GGGCGCCCCC pattern
#         2. Shuffle each instance independently
#         3. Maintains GC content within each motif
        
#         Args:
#             slice_tensor: [1, 4, 2048] tensor
#             seed: random seed for reproducibility
        
#         Returns:
#             Modified tensor with Plagl1 motifs shuffled
#         """
#         device = slice_tensor.device
        
#         # Convert to sequence
#         sequence = self.tensor_to_sequence(slice_tensor)
#         seq_list = list(sequence)
        
#         # Find Plagl1 motif instances
#         motif_instances = self.find_plagl1_motifs(sequence)
        
#         # Shuffle each motif instance
#         if seed is not None:
#             np.random.seed(seed)
        
#         for start, end in motif_instances:
#             motif_bases = seq_list[start:end]
#             np.random.shuffle(motif_bases)
#             seq_list[start:end] = motif_bases
        
#         # Convert back to tensor
#         modified_sequence = ''.join(seq_list)
#         modified_tensor = self.sequence_to_tensor(modified_sequence, device)
        
#         return modified_tensor.unsqueeze(0)  # Add batch dimension back
    
#     def count_motifs(self, sequence):
#         """Count Plagl1 motifs and related patterns."""
#         patterns = {
#             'Plagl1_core_8bp': len(re.findall(r'GGGCGCCC', sequence)),  # 8bp core
#             'Plagl1_full_10bp': len(re.findall(r'GGGCGCCCCC', sequence)),  # 10bp version
#             'Plagl1_rev_comp': len(re.findall(r'GGGCGCCC', sequence)),  # Reverse complement of core
#             'G3+': len(re.findall(r'G{3,}', sequence)),
#             'C3+': len(re.findall(r'C{3,}', sequence)),
#             'G4+': len(re.findall(r'G{4,}', sequence)),
#             'C4+': len(re.findall(r'C{4,}', sequence)),
#             'G5+': len(re.findall(r'G{5,}', sequence)),
#             'C5+': len(re.findall(r'C{5,}', sequence)),
#         }
#         return patterns

#     def __getitem__(self, idx):
#         row = self.coords.iloc[idx]
#         chrom = row["chrom"]
#         start = row["centered_start"]
#         end = row["centered_end"]
        
#         device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        
#         X = torch.load(f"{self.init_seq_path}{chrom}_{start}_{end}_X.pt", weights_only=True, map_location=device)
#         slice_edited = torch.load(f"{self.slice_path}{chrom}_{start}_{end}_slice.pt", weights_only=True, map_location=device)
        
#         # Shuffle Plagl1 (UP00088_1) motif instances
#         slice_modified = self.shuffle_plagl1_motifs(slice_edited, seed=idx)
        
#         if idx < 3:
#             # Debug output for first few sequences
#             orig_seq = self.tensor_to_sequence(slice_edited)
#             mod_seq = self.tensor_to_sequence(slice_modified)
            
#             orig_patterns = self.count_motifs(orig_seq)
#             mod_patterns = self.count_motifs(mod_seq)
            
#             print(f"\nSequence {idx}:")
#             print(f"  Plagl1 motifs found:")
#             print(f"    8bp core (GGGCGCCC):   {orig_patterns['Plagl1_core_8bp']} instances")
#             print(f"    10bp full (GGGCGCCCCC): {orig_patterns['Plagl1_full_10bp']} instances")
            
#             if orig_patterns['Plagl1_core_8bp'] > 0:
#                 # Show example of shuffled motif
#                 for match in re.finditer(r'GGGCGCCC', orig_seq):
#                     print(f"    Example 8bp motif shuffled:")
#                     print(f"      Original: {orig_seq[match.start():match.end()]}")
#                     print(f"      Shuffled: {mod_seq[match.start():match.end()]}")
#                     break
            
#             print(f"  Homopolymers before/after shuffling:")
#             print(f"    G3+: {orig_patterns['G3+']:2d} → {mod_patterns['G3+']:2d} ({mod_patterns['G3+'] - orig_patterns['G3+']:+2d})")
#             print(f"    C3+: {orig_patterns['C3+']:2d} → {mod_patterns['C3+']:2d} ({mod_patterns['C3+'] - orig_patterns['C3+']:+2d})")
#             print(f"    G4+: {orig_patterns['G4+']:2d} → {mod_patterns['G4+']:2d} ({mod_patterns['G4+'] - orig_patterns['G4+']:+2d})")
#             print(f"    C4+: {orig_patterns['C4+']:2d} → {mod_patterns['C4+']:2d} ({mod_patterns['C4+'] - orig_patterns['C4+']:+2d})")
            
#             # Verify GC content unchanged
#             orig_gc = (orig_seq.count('G') + orig_seq.count('C')) / len(orig_seq)
#             mod_gc = (mod_seq.count('G') + mod_seq.count('C')) / len(mod_seq)
#             print(f"  GC content: {orig_gc:.4f} → {mod_gc:.4f} (diff: {mod_gc - orig_gc:.6f})")
        
#         edit_start = (self.slice + self.cropping) * self.bin_size
#         edit_end = edit_start + self.bin_size
        
#         editedX = X.clone()
#         editedX[:, :, edit_start:edit_end] = slice_modified
        
#         editedX = editedX.squeeze(0)
        
#         return editedX

In [ ]:
# # ═══════════════════════════════════════════════════════════════════════════════
# # B-BOX DETECTION
# # ═══════════════════════════════════════════════════════════════════════════════

# # B-box consensus: RGTTCRANTCC
# # IUPAC: R=[AG], N=[ACGT]
# # We encode this as a list of allowed bases per position
# BBOX_CONSENSUS = [
#     {"A", "G"},         # R
#     {"G"},              # G
#     {"T"},              # T
#     {"T"},              # T
#     {"C"},              # C
#     {"A", "G"},         # R
#     {"A", "C", "G", "T"},  # N
#     {"A", "G"},         # R (sometimes written as N, but canonical is R)
#     {"T"},              # T
#     {"C"},              # C
#     {"C"},              # C
# ]

# BBOX_LEN = len(BBOX_CONSENSUS)  # 11


# def count_mismatches(seq, consensus=BBOX_CONSENSUS):
#     """Count mismatches between a sequence and the B-box consensus."""
#     if len(seq) < len(consensus):
#         return len(consensus)
#     mismatches = 0
#     for i, allowed in enumerate(consensus):
#         if seq[i].upper() not in allowed:
#             mismatches += 1
#     return mismatches

# def find_bbox_hits(seq_str, max_mismatches=2):
#     """
#     Scan a DNA string for B-box-like motifs using fuzzy matching.

#     Args:
#         seq_str: DNA sequence string (ACGT)
#         max_mismatches: maximum number of mismatches to B-box consensus (default 2)

#     Returns:
#         List of (start, end) tuples for each hit (0-indexed, end exclusive)
#     """
#     hits = []
#     for i in range(len(seq_str) - BBOX_LEN + 1):
#         subseq = seq_str[i:i + BBOX_LEN]
#         if count_mismatches(subseq) <= max_mismatches:
#             hits.append((i, i + BBOX_LEN))
#     # Merge overlapping hits
#     if not hits:
#         return hits
#     merged = [hits[0]]
#     for start, end in hits[1:]:
#         if start <= merged[-1][1]:
#             merged[-1] = (merged[-1][0], max(merged[-1][1], end))
#         else:
#             merged.append((start, end))
#     return merged

In [ ]:
# # ═══════════════════════════════════════════════════════════════════════════════
# # FILTER 21 MOTIF DETECTION
# # ═══════════════════════════════════════════════════════════════════════════════

# FILTER21_CONSENSUS = "CTGAAGACAGCTACA"
# FILTER21_LEN = len(FILTER21_CONSENSUS)  # 15


# def count_mismatches_str(seq, consensus=FILTER21_CONSENSUS):
#     """Count mismatches between a sequence and the filter 21 consensus."""
#     return sum(1 for a, b in zip(seq.upper(), consensus) if a != b)


# def find_filter21_hits(seq_str, max_mismatches=3):
#     """
#     Scan a DNA string for filter 21 motif using fuzzy matching.
#     Checks both forward and reverse complement.

#     Args:
#         seq_str: DNA sequence string (ACGT)
#         max_mismatches: max mismatches allowed (default 3)

#     Returns:
#         List of (start, end) tuples for each hit (0-indexed, end exclusive)
#     """
#     hits = []

#     # Also check reverse complement
#     rc = reverse_complement(FILTER21_CONSENSUS)

#     for i in range(len(seq_str) - FILTER21_LEN + 1):
#         subseq = seq_str[i:i + FILTER21_LEN].upper()
#         mm_fwd = count_mismatches_str(subseq, FILTER21_CONSENSUS)
#         mm_rc = count_mismatches_str(subseq, rc)
#         if min(mm_fwd, mm_rc) <= max_mismatches:
#             hits.append((i, i + FILTER21_LEN))

#     # Merge overlapping hits
#     if not hits:
#         return hits
#     merged = [hits[0]]
#     for start, end in hits[1:]:
#         if start <= merged[-1][1]:
#             merged[-1] = (merged[-1][0], max(merged[-1][1], end))
#         else:
#             merged.append((start, end))
#     return merged


# def reverse_complement(seq):
#     """Return reverse complement of a DNA string."""
#     comp = {"A": "T", "T": "A", "C": "G", "G": "C",
#             "N": "N", "R": "Y", "Y": "R"}
#     return "".join(comp.get(b, "N") for b in reversed(seq.upper()))


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FILTER 104 MOTIF DETECTION
# ═══════════════════════════════════════════════════════════════════════════════

FILTER104_CONSENSUS = "CCCTCTTCTGG"
FILTER104_LEN = len(FILTER104_CONSENSUS)  # 15


def reverse_complement(seq):
    """Return reverse complement of a DNA string."""
    comp = {"A": "T", "T": "A", "C": "G", "G": "C", "N": "N"}
    return "".join(comp.get(b, "N") for b in reversed(seq.upper()))


def count_mismatches_str(seq, consensus):
    """Count mismatches between two equal-length strings."""
    return sum(1 for a, b in zip(seq.upper(), consensus) if a != b)


def find_filter104_hits(seq_str, max_mismatches=3):
    """
    Scan a DNA string for filter 104 motif using fuzzy matching.
    Checks both forward and reverse complement.

    Args:
        seq_str: DNA sequence string (ACGT)
        max_mismatches: max mismatches allowed (default 3)

    Returns:
        List of (start, end) tuples for each hit (0-indexed, end exclusive)
    """
    hits = []
    rc = reverse_complement(FILTER104_CONSENSUS)

    for i in range(len(seq_str) - FILTER104_LEN + 1):
        subseq = seq_str[i:i + FILTER104_LEN].upper()
        mm_fwd = count_mismatches_str(subseq, FILTER104_CONSENSUS)
        mm_rc = count_mismatches_str(subseq, rc)
        if min(mm_fwd, mm_rc) <= max_mismatches:
            hits.append((i, i + FILTER104_LEN))

    # Merge overlapping hits
    if not hits:
        return hits
    merged = [hits[0]]
    for start, end in hits[1:]:
        if start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))
    return merged


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ONE-HOT ↔ SEQUENCE CONVERSION
# ═══════════════════════════════════════════════════════════════════════════════

INDEX_TO_BASE = {0: "A", 1: "C", 2: "G", 3: "T"}
BASE_TO_INDEX = {"A": 0, "C": 1, "G": 2, "T": 3}


def onehot_to_seq(onehot):
    """
    Convert one-hot tensor to DNA string.
    onehot: shape (4, L) — channels-first
    """
    indices = onehot.argmax(dim=0)  # (L,)
    return "".join(INDEX_TO_BASE[i.item()] for i in indices)


def seq_to_onehot(seq_str, device=None):
    """
    Convert DNA string to one-hot tensor, shape (4, L).
    """
    L = len(seq_str)
    oh = torch.zeros(4, L)
    for i, base in enumerate(seq_str.upper()):
        if base in BASE_TO_INDEX:
            oh[BASE_TO_INDEX[base], i] = 1.0
        else:
            oh[:, i] = 0.25  # ambiguous
    if device is not None:
        oh = oh.to(device)
    return oh

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SHUFFLING
# ═══════════════════════════════════════════════════════════════════════════════

def shuffle_regions(seq_str, regions):
    """
    Randomly shuffle nucleotides within specified regions of a DNA string.

    Args:
        seq_str: DNA sequence string
        regions: list of (start, end) tuples

    Returns:
        New DNA string with specified regions shuffled
    """
    seq_list = list(seq_str)
    for start, end in regions:
        subseq = seq_list[start:end]
        random.shuffle(subseq)
        seq_list[start:end] = subseq
    return "".join(seq_list)

In [ ]:
# # ═══════════════════════════════════════════════════════════════════════════════
# # DATASET
# # ═══════════════════════════════════════════════════════════════════════════════

# class BoundaryGenerationDataset(Dataset):
#     """
#     Like BoundaryGenerationDataset, but B-box motifs within the SINE B2
#     slice are shuffled before insertion.

#     This tests whether the Pol III Box B promoter element is required for
#     the model's prediction of CTCF-mediated boundary formation at
#     SINE B2 elements.

#     Args:
#         coord_df: DataFrame with chrom, centered_start, centered_end columns
#         init_seq_path: path prefix for initial sequence tensors
#         slice_path: path prefix for SINE B2 slice tensors
#         slice: number of bins for slice offset (default 256)
#         cropping: cropping bins (default 64)
#         bin_size: genomic bin size in bp (default 2048)
#         max_mismatches: max mismatches for B-box detection (default 2)
#         seed: random seed for reproducibility (default None)
#     """

#     def __init__(self, coord_df, init_seq_path, slice_path,
#                  slice=256, cropping=64, bin_size=2048,
#                  max_mismatches=2, seed=None):
#         self.coords = coord_df
#         self.init_seq_path = init_seq_path
#         self.slice_path = slice_path
#         self.slice = slice
#         self.cropping = cropping
#         self.bin_size = bin_size
#         self.max_mismatches = max_mismatches
#         self.seed = seed

#     def __len__(self):
#         return len(self.coords)

#     def __getitem__(self, idx):
#         row = self.coords.iloc[idx]
#         chrom = row["chrom"]
#         start = row["centered_start"]
#         end = row["centered_end"]
#         fold = row["fold"]
        
#         device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

#         X = torch.load(
#             f"{self.init_seq_path}/fold{fold}/{chrom}_{start}_{end}_X.pt",
#             weights_only=True, map_location=device
#         )
#         slice_tensor = torch.load(
#             f"{self.slice_path}/fold{fold}/{chrom}_{start}_{end}_slice.pt",
#             weights_only=True, map_location=device
#         )

#         # ── Shuffle B-box motifs within the slice ────────────────────────
#         # slice_tensor shape: (1, 4, L) or (4, L)
#         if slice_tensor.dim() == 3:
#             slice_2d = slice_tensor.squeeze(0)  # (4, L)
#         else:
#             slice_2d = slice_tensor

#         # Convert to sequence string
#         seq_str = onehot_to_seq(slice_2d)

#         # Find B-box hits
#         bbox_hits = find_bbox_hits(seq_str, max_mismatches=self.max_mismatches)

#         # Shuffle if any hits found
#         if bbox_hits:
#             if self.seed is not None:
#                 random.seed(self.seed + idx)
#             shuffled_seq = shuffle_regions(seq_str, bbox_hits)
#             shuffled_tensor = seq_to_onehot(shuffled_seq, device=device)

#             # Reconstruct slice_tensor shape
#             if slice_tensor.dim() == 3:
#                 slice_tensor = shuffled_tensor.unsqueeze(0)
#             else:
#                 slice_tensor = shuffled_tensor

#         # ── Insert (shuffled) slice into full sequence ───────────────────
#         edit_start = (self.slice + self.cropping) * self.bin_size
#         edit_end = edit_start + self.bin_size

#         editedX = X.clone()
#         editedX[:, :, edit_start:edit_end] = slice_tensor

#         editedX = editedX.squeeze(0)

#         return editedX


In [ ]:
# # ═══════════════════════════════════════════════════════════════════════════════
# # DATASET
# # ═══════════════════════════════════════════════════════════════════════════════

# class BoundaryGenerationDataset(Dataset):
#     """
#     Like BoundaryGenerationDataset, but filter 21 motif instances
#     (CTGAAGACAGCTACA — the B2-specific CTCF feature) within the SINE B2
#     slice are shuffled before insertion.

#     Args:
#         coord_df: DataFrame with chrom, centered_start, centered_end columns
#         init_seq_path: path prefix for initial sequence tensors
#         slice_path: path prefix for SINE B2 slice tensors
#         slice: number of bins for slice offset (default 256)
#         cropping: cropping bins (default 64)
#         bin_size: genomic bin size in bp (default 2048)
#         max_mismatches: max mismatches for motif detection (default 3)
#         seed: random seed for reproducibility (default None)
#     """

#     def __init__(self, coord_df, init_seq_path, slice_path,
#                  slice=256, cropping=64, bin_size=2048,
#                  max_mismatches=3, seed=None):
#         self.coords = coord_df
#         self.init_seq_path = init_seq_path
#         self.slice_path = slice_path
#         self.slice = slice
#         self.cropping = cropping
#         self.bin_size = bin_size
#         self.max_mismatches = max_mismatches
#         self.seed = seed

#     def __len__(self):
#         return len(self.coords)

#     def __getitem__(self, idx):
#         row = self.coords.iloc[idx]
#         chrom = row["chrom"]
#         start = row["centered_start"]
#         end = row["centered_end"]
#         fold = row["fold"]
        
#         device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

#         X = torch.load(
#             f"{self.init_seq_path}/fold{fold}/{chrom}_{start}_{end}_X.pt",
#             weights_only=True, map_location=device
#         )
#         slice_tensor = torch.load(
#             f"{self.slice_path}/fold{fold}/{chrom}_{start}_{end}_slice.pt",
#             weights_only=True, map_location=device
#         )

#         # ── Shuffle filter 21 motifs within the slice ────────────────────
#         if slice_tensor.dim() == 3:
#             slice_2d = slice_tensor.squeeze(0)
#         else:
#             slice_2d = slice_tensor

#         seq_str = onehot_to_seq(slice_2d)
#         hits = find_filter21_hits(seq_str, max_mismatches=self.max_mismatches)

#         if hits:
#             if self.seed is not None:
#                 random.seed(self.seed + idx)
#             shuffled_seq = shuffle_regions(seq_str, hits)
#             shuffled_tensor = seq_to_onehot(shuffled_seq, device=device)

#             if slice_tensor.dim() == 3:
#                 slice_tensor = shuffled_tensor.unsqueeze(0)
#             else:
#                 slice_tensor = shuffled_tensor

#         # ── Insert (shuffled) slice into full sequence ───────────────────
#         edit_start = (self.slice + self.cropping) * self.bin_size
#         edit_end = edit_start + self.bin_size

#         editedX = X.clone()
#         editedX[:, :, edit_start:edit_end] = slice_tensor

#         editedX = editedX.squeeze(0)

#         return editedX

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════════════════════════

class BoundaryGenerationDataset(Dataset):
    """
    Like BoundaryGenerationDataset, but filter 104 motif instances
    (CCCTCTTCTGGTGTG — the B2-derived CTCF variant that newly added
    CTCFs match better than preexisting ones) within the SINE B2
    slice are shuffled before insertion.

    Args:
        coord_df: DataFrame with chrom, centered_start, centered_end columns
        init_seq_path: path prefix for initial sequence tensors
        slice_path: path prefix for SINE B2 slice tensors
        slice: number of bins for slice offset (default 256)
        cropping: cropping bins (default 64)
        bin_size: genomic bin size in bp (default 2048)
        max_mismatches: max mismatches for motif detection (default 3)
        seed: random seed for reproducibility (default None)
    """

    def __init__(self, coord_df, init_seq_path, slice_path,
                 slice=256, cropping=64, bin_size=2048,
                 max_mismatches=3, seed=None):
        self.coords = coord_df
        self.init_seq_path = init_seq_path
        self.slice_path = slice_path
        self.slice = slice
        self.cropping = cropping
        self.bin_size = bin_size
        self.max_mismatches = max_mismatches
        self.seed = seed

    def __len__(self):
        return len(self.coords)

    def __getitem__(self, idx):
        row = self.coords.iloc[idx]
        chrom = row["chrom"]
        start = row["centered_start"]
        end = row["centered_end"]
        fold = row["fold"]

        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

        X = torch.load(
            f"{self.init_seq_path}/fold{fold}/{chrom}_{start}_{end}_X.pt",
            weights_only=True, map_location=device
        )
        slice_tensor = torch.load(
            f"{self.slice_path}/fold{fold}/{chrom}_{start}_{end}_slice.pt",
            weights_only=True, map_location=device
        )

        # ── Shuffle filter 104 motifs within the slice ───────────────────
        if slice_tensor.dim() == 3:
            slice_2d = slice_tensor.squeeze(0)
        else:
            slice_2d = slice_tensor

        seq_str = onehot_to_seq(slice_2d)
        hits = find_filter104_hits(seq_str, max_mismatches=self.max_mismatches)

        if hits:
            if self.seed is not None:
                random.seed(self.seed + idx)
            shuffled_seq = shuffle_regions(seq_str, hits)
            shuffled_tensor = seq_to_onehot(shuffled_seq, device=device)

            if slice_tensor.dim() == 3:
                slice_tensor = shuffled_tensor.unsqueeze(0)
            else:
                slice_tensor = shuffled_tensor

        # ── Insert (shuffled) slice into full sequence ───────────────────
        edit_start = (self.slice + self.cropping) * self.bin_size
        edit_end = edit_start + self.bin_size

        editedX = X.clone()
        editedX[:, :, edit_start:edit_end] = slice_tensor

        editedX = editedX.squeeze(0)

        return editedX

In [ ]:
# # B-box: IUPAC RGTTCRANTCC
# BBOX_CONSENSUS = [
#     {"A", "G"},             # R
#     {"G"},                  # G
#     {"T"},                  # T
#     {"T"},                  # T
#     {"C"},                  # C
#     {"A", "G"},             # R
#     {"A", "C", "G", "T"},   # N
#     {"A", "G"},             # R
#     {"T"},                  # T
#     {"C"},                  # C
#     {"C"},                  # C
# ]
# BBOX_LEN = 11

# FILTER21_CONSENSUS = "CTGAAGACAGCTACA"
# FILTER104_CONSENSUS = "CCCTCTTCTGG"

# COMP = {"A": "T", "T": "A", "C": "G", "G": "C", "N": "N"}


# def reverse_complement(seq):
#     return "".join(COMP.get(b, "N") for b in reversed(seq.upper()))


# FILTER21_RC = reverse_complement(FILTER21_CONSENSUS)
# FILTER104_RC = reverse_complement(FILTER104_CONSENSUS)

In [ ]:
# # ═══════════════════════════════════════════════════════════════════════════════
# # MOTIF DETECTION
# # ═══════════════════════════════════════════════════════════════════════════════

# def bbox_mismatches(seq):
#     """Count mismatches to B-box IUPAC consensus."""
#     return sum(1 for i, allowed in enumerate(BBOX_CONSENSUS)
#                if seq[i].upper() not in allowed)


# def str_mismatches(seq, consensus):
#     """Count mismatches between two equal-length strings."""
#     return sum(1 for a, b in zip(seq.upper(), consensus) if a != b)


# def find_all_hits(seq_str, max_mismatches=3):
#     """
#     Scan for all three motifs simultaneously. Returns merged (start, end)
#     intervals with motif labels.

#     Args:
#         seq_str: DNA sequence string
#         max_mismatches: max mismatches per motif (default 3)

#     Returns:
#         List of (start, end) tuples — merged across all motifs
#         Dict of per-motif hit counts for diagnostics
#     """
#     hits = []
#     counts = {"bbox": 0, "filter21": 0, "filter104": 0}

#     # B-box (11bp)
#     for i in range(len(seq_str) - BBOX_LEN + 1):
#         sub = seq_str[i:i + BBOX_LEN]
#         if bbox_mismatches(sub) <= max_mismatches:
#             hits.append((i, i + BBOX_LEN))
#             counts["bbox"] += 1

#     # Filter 21 (15bp, both strands)
#     for i in range(len(seq_str) - 15 + 1):
#         sub = seq_str[i:i + 15].upper()
#         if min(str_mismatches(sub, FILTER21_CONSENSUS),
#                str_mismatches(sub, FILTER21_RC)) <= max_mismatches:
#             hits.append((i, i + 15))
#             counts["filter21"] += 1

#     # Filter 104 (15bp, both strands)
#     for i in range(len(seq_str) - 15 + 1):
#         sub = seq_str[i:i + 15].upper()
#         if min(str_mismatches(sub, FILTER104_CONSENSUS),
#                str_mismatches(sub, FILTER104_RC)) <= max_mismatches:
#             hits.append((i, i + 15))
#             counts["filter104"] += 1

#     # Sort and merge overlapping intervals
#     if not hits:
#         return [], counts
#     hits.sort()
#     merged = [hits[0]]
#     for start, end in hits[1:]:
#         if start <= merged[-1][1]:
#             merged[-1] = (merged[-1][0], max(merged[-1][1], end))
#         else:
#             merged.append((start, end))

#     return merged, counts


In [ ]:
# # ═══════════════════════════════════════════════════════════════════════════════
# # DATASET
# # ═══════════════════════════════════════════════════════════════════════════════

# class BoundaryGenerationDataset(Dataset):
#     """
#     Like BoundaryGenerationDataset, but ALL THREE B2-context motifs
#     (B-box, filter 21, filter 104) are detected and shuffled within
#     the SINE B2 slice before insertion.

#     This is the "full disruption" condition — if boundary predictions
#     collapse here but not in the single-motif conditions, it indicates
#     redundancy among the features.

#     Args:
#         coord_df: DataFrame with chrom, centered_start, centered_end columns
#         init_seq_path: path prefix for initial sequence tensors
#         slice_path: path prefix for SINE B2 slice tensors
#         slice: number of bins for slice offset (default 256)
#         cropping: cropping bins (default 64)
#         bin_size: genomic bin size in bp (default 2048)
#         max_mismatches: max mismatches per motif (default 3)
#         seed: random seed for reproducibility (default None)
#     """

#     def __init__(self, coord_df, init_seq_path, slice_path,
#                  slice=256, cropping=64, bin_size=2048,
#                  max_mismatches=3, seed=None):
#         self.coords = coord_df
#         self.init_seq_path = init_seq_path
#         self.slice_path = slice_path
#         self.slice = slice
#         self.cropping = cropping
#         self.bin_size = bin_size
#         self.max_mismatches = max_mismatches
#         self.seed = seed

#     def __len__(self):
#         return len(self.coords)

#     def __getitem__(self, idx):
#         row = self.coords.iloc[idx]
#         chrom = row["chrom"]
#         start = row["centered_start"]
#         end = row["centered_end"]

#         device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

#         X = torch.load(
#             f"{self.init_seq_path}{chrom}_{start}_{end}_X.pt",
#             weights_only=True, map_location=device
#         )
#         slice_tensor = torch.load(
#             f"{self.slice_path}{chrom}_{start}_{end}_slice.pt",
#             weights_only=True, map_location=device
#         )

#         # ── Shuffle all three motifs within the slice ────────────────────
#         if slice_tensor.dim() == 3:
#             slice_2d = slice_tensor.squeeze(0)
#         else:
#             slice_2d = slice_tensor

#         seq_str = onehot_to_seq(slice_2d)
#         hits, _ = find_all_hits(seq_str, max_mismatches=self.max_mismatches)

#         if hits:
#             if self.seed is not None:
#                 random.seed(self.seed + idx)
#             shuffled_seq = shuffle_regions(seq_str, hits)
#             shuffled_tensor = seq_to_onehot(shuffled_seq, device=device)

#             if slice_tensor.dim() == 3:
#                 slice_tensor = shuffled_tensor.unsqueeze(0)
#             else:
#                 slice_tensor = shuffled_tensor

#         # ── Insert (shuffled) slice into full sequence ───────────────────
#         edit_start = (self.slice + self.cropping) * self.bin_size
#         edit_end = edit_start + self.bin_size

#         editedX = X.clone()
#         editedX[:, :, edit_start:edit_end] = slice_tensor

#         editedX = editedX.squeeze(0)

#         return editedX

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
model = SeqNN()
model_path = (
    "/home1/smaruj/pytorch_akita/models/finetuned/mouse/Hsieh2019_mESC/checkpoints/Akita_v2_mouse_Hsieh2019_mESC_model0_finetuned.pth"
)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()


In [ ]:
def set_diag(matrix, value, k):
    """Set diagonal `k` of a matrix to `value`."""
    rows, cols = matrix.shape
    for i in range(rows):
        if 0 <= i + k < cols:
            matrix[i, i + k] = value

def from_upper_triu_batch(batch_vectors, matrix_len=512, num_diags=2):
    """Convert a batch of upper-triangular vectors into symmetric matrices with np.nan on diagonals."""
    if isinstance(batch_vectors, torch.Tensor):
        batch_vectors = batch_vectors.detach().cpu().numpy()

    batch_size = len(batch_vectors)
    matrices = np.zeros((batch_size, matrix_len, matrix_len), dtype=np.float32)

    triu_indices = np.triu_indices(matrix_len, num_diags)

    for i in range(batch_size):
        matrices[i][triu_indices] = batch_vectors[i][0,:]
        # Mirror to lower triangle
        matrices[i] = matrices[i] + matrices[i].T

        # Set diagonals to np.nan
        for k in range(-num_diags + 1, num_diags):
            set_diag(matrices[i], np.nan, k)

    return matrices  # shape: [B, 512, 512]

In [ ]:
orig_dataset = OriginalDataset(df, f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/ohe_X")

orig_loader = DataLoader(orig_dataset, batch_size=4, shuffle=False)

In [ ]:
## for the control

# edited_dataset = BoundaryGenerationDataset(df, 
#                                     f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/ohe_X", 
#                                     f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/results_control_repeated")

In [ ]:
# edited_dataset = BoundaryGenerationDataset(df, 
#                                 f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/ohe_X", 
#                                 f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/results_repeated")

edited_dataset = BoundaryGenerationDataset(df, 
                                f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/ohe_X", 
                                f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/results_repeated",
                                max_mismatches=3)

edited_loader = DataLoader(edited_dataset, batch_size=4, shuffle=False)

In [ ]:
# df

In [ ]:
slice = 256
cropping = 64
bin_size = 2048

edit_start = (slice + cropping) * bin_size
edit_end = edit_start + bin_size

In [ ]:
preds_all_orig = []
preds_all_reversed = []
urq_mean_values = []

model.eval()
with torch.no_grad():
    for orig_batch, edited_batch in zip(orig_loader, edited_loader):
        orig_batch = orig_batch.to(device)
        edited_batch = edited_batch.to(device)

        preds_orig = model(orig_batch).cpu()
        preds_edited = model(edited_batch).cpu()

        preds_all_orig.extend(preds_orig)
        preds_all_reversed.extend(preds_edited)
        
        orig_maps = from_upper_triu_batch(preds_orig)
        edited_maps = from_upper_triu_batch(preds_edited)
        
        urq_mean = np.nanmean(edited_maps[:, 0:250, 260:512], axis=(1, 2))
        urq_mean_values.extend(urq_mean)

In [ ]:
preds_orig = model(orig_batch).cpu()
orig_maps = from_upper_triu_batch(preds_orig)
urq_mean = np.nanmean(edited_maps[:, 0:250, 260:512], axis=(1, 2))

In [ ]:
# preds_all_orig = []
# preds_all_reversed = []
# urq_mean_values = []

# model.eval()
# with torch.no_grad():
#     for orig_batch, edited_batch in zip(orig_loader, edited_loader):
#         orig_batch = orig_batch.to(device)
#         edited_batch = edited_batch.to(device)

#         preds_orig = model(orig_batch).cpu()
#         preds_edited = model(edited_batch).cpu()

#         preds_all_orig.extend(preds_orig)
#         preds_all_reversed.extend(preds_edited)
        
#         orig_maps = from_upper_triu_batch(preds_orig)
#         edited_maps = from_upper_triu_batch(preds_edited)
        
#         urq_mean = np.nanmean(orig_maps[:, 0:250, 260:512], axis=(1, 2))
#         urq_mean_values.extend(urq_mean)

In [ ]:
df.columns

In [ ]:
df["URQ_no_sineB2_ctcf"] = urq_mean_values

In [ ]:
df["URQ_no_sineB2_ctcf"].mean()

In [ ]:
# plot_df[["URQ_result", "URQ_shuffle_new"]]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df

In [ ]:
df.to_csv(f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/results_repeated/only_successful_seqs_with_results.tsv", sep="\t", index=False)

In [ ]:
df.columns

In [ ]:
# Define the columns to plot
urq_columns = [
    'URQ_init',
    'URQ_control',
    'URQ_result', 
    'URQ_shuffle_new_CTCFs',
    'URQ_no_bbox',
    # 'URQ_no_sineB2_tail',
    'URQ_no_homopolymers'
]

# Create the boxplot with filled boxes
fig, ax = plt.subplots(figsize=(14, 6))

bp = ax.boxplot(
    [df[col].dropna() for col in urq_columns],
    labels=urq_columns,
    patch_artist=True,
    widths=0.6
)

# Better color scheme - use tab20 which has 20 distinct colors
colors = plt.cm.tab20(np.linspace(0, 1, len(urq_columns)))

# OR manually define colors for better control:
# colors = [
#     '#8dd3c7', '#ffffb3', '#bebada', '#fb8072', '#80b1d3', '#fdb462',
#     '#b3de69', '#fccde5', '#d9d9d9', '#bc80bd', '#ccebc5', '#ffed6f',
#     '#e5c494', '#a6cee3'
# ]

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Customize the plot
ax.set_xticklabels(urq_columns, rotation=45, ha='right')
ax.set_ylabel('URQ Value', fontsize=12)
ax.set_title('URQ Values Across Different Conditions', fontsize=14)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Print basic statistics for each URQ column
print("Basic Statistics for URQ Columns")
print("=" * 80)

for col in urq_columns:
    print(f"\n{col}:")
    print(f"  Mean:   {df[col].mean():.6f}")
    print(f"  Median: {df[col].median():.6f}")
    print(f"  Std:    {df[col].std():.6f}")
    print(f"  Min:    {df[col].min():.6f}")
    print(f"  Max:    {df[col].max():.6f}")

# Or use pandas describe for a more comprehensive view
print("\n" + "=" * 80)
print("\nComprehensive Statistics:")
print(df[urq_columns].describe().T)

In [ ]:
def tensor_to_sequence(seq_tensor):
    """Convert one-hot tensor to DNA sequence string."""
    if seq_tensor.dim() == 3:
        seq_tensor = seq_tensor.squeeze(0)
    
    nucleotides = ['A', 'C', 'G', 'T']
    indices = torch.argmax(seq_tensor, dim=0)
    sequence = ''.join([nucleotides[i] for i in indices])
    return sequence

In [ ]:
def find_homopolymers(sequence, min_length=3):
    """
    Find all G and C homopolymers in a sequence.
    
    Args:
        sequence: DNA sequence string
        min_length: minimum homopolymer length
    
    Returns:
        List of (position, length, base_type) tuples
    """
    homopolymers = []
    
    # Find G homopolymers
    for match in re.finditer(f'G{{{min_length},}}', sequence):
        homopolymers.append({
            'start': match.start(),
            'end': match.end(),
            'length': match.end() - match.start(),
            'base': 'G',
            'center': (match.start() + match.end()) / 2
        })
    
    # Find C homopolymers
    for match in re.finditer(f'C{{{min_length},}}', sequence):
        homopolymers.append({
            'start': match.start(),
            'end': match.end(),
            'length': match.end() - match.start(),
            'base': 'C',
            'center': (match.start() + match.end()) / 2
        })
    
    # Sort by position
    homopolymers.sort(key=lambda x: x['start'])
    
    return homopolymers

In [ ]:
def calculate_distances(homopolymers):
    """
    Calculate distances between consecutive homopolymers.
    
    Returns:
        List of distances (in bp)
    """
    if len(homopolymers) < 2:
        return []
    
    distances = []
    for i in range(len(homopolymers) - 1):
        # Distance from end of current to start of next
        distance = homopolymers[i+1]['start'] - homopolymers[i]['end']
        distances.append(distance)
    
    return distances

In [ ]:
def analyze_homopolymer_distribution(coord_df, slice_path, min_length=3):
    """
    Analyze homopolymer distribution across all edited sequences.
    
    Args:
        coord_df: DataFrame with sequence coordinates
        slice_path: Path to edited slices
        min_length: minimum homopolymer length
    
    Returns:
        Dictionary with statistics
    """
    all_distances = []
    all_positions = []
    all_lengths = []
    all_bases = []
    homopolymers_per_seq = []
    
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    
    print(f"Analyzing {len(coord_df)} sequences...")
    
    for idx, row in coord_df.iterrows():
        if idx % 50 == 0:
            print(f"  Processing sequence {idx}/{len(coord_df)}...")
        
        chrom = row["chrom"]
        start = row["centered_start"]
        end = row["centered_end"]
        fold = row["fold"]
        
        # Load edited slice
        slice_path_full = f"{slice_path}/fold{fold}/{chrom}_{start}_{end}_slice.pt"
        slice_tensor = torch.load(slice_path_full, weights_only=True, map_location=device)
        
        # Convert to sequence
        sequence = tensor_to_sequence(slice_tensor)
        
        # Find homopolymers
        homopolymers = find_homopolymers(sequence, min_length)
        
        # Store statistics
        homopolymers_per_seq.append(len(homopolymers))
        
        for homo in homopolymers:
            all_positions.append(homo['center'])
            all_lengths.append(homo['length'])
            all_bases.append(homo['base'])
        
        # Calculate distances
        distances = calculate_distances(homopolymers)
        all_distances.extend(distances)
    
    return {
        'distances': np.array(all_distances),
        'positions': np.array(all_positions),
        'lengths': np.array(all_lengths),
        'bases': all_bases,
        'count_per_seq': np.array(homopolymers_per_seq)
    }

In [ ]:
def print_summary_statistics(stats):
    """Print summary statistics."""
    print("\n" + "="*70)
    print("HOMOPOLYMER DISTRIBUTION SUMMARY")
    print("="*70)
    
    print(f"\nTotal homopolymers found: {len(stats['positions'])}")
    print(f"  G homopolymers: {stats['bases'].count('G')}")
    print(f"  C homopolymers: {stats['bases'].count('C')}")
    
    print(f"\nHomopolymers per sequence:")
    print(f"  Mean: {np.mean(stats['count_per_seq']):.2f}")
    print(f"  Median: {np.median(stats['count_per_seq']):.0f}")
    print(f"  Min: {np.min(stats['count_per_seq']):.0f}")
    print(f"  Max: {np.max(stats['count_per_seq']):.0f}")
    
    print(f"\nHomopolymer lengths:")
    print(f"  Mean: {np.mean(stats['lengths']):.2f} bp")
    print(f"  Median: {np.median(stats['lengths']):.0f} bp")
    print(f"  Min: {np.min(stats['lengths']):.0f} bp")
    print(f"  Max: {np.max(stats['lengths']):.0f} bp")
    
    if len(stats['distances']) > 0:
        print(f"\nDistances between consecutive homopolymers:")
        print(f"  Mean: {np.mean(stats['distances']):.2f} bp")
        print(f"  Median: {np.median(stats['distances']):.0f} bp")
        print(f"  Min: {np.min(stats['distances']):.0f} bp")
        print(f"  Max: {np.max(stats['distances']):.0f} bp")
        print(f"  Std: {np.std(stats['distances']):.2f} bp")
        
        # Percentiles
        print(f"\n  Percentiles:")
        for p in [10, 25, 50, 75, 90]:
            print(f"    {p}th: {np.percentile(stats['distances'], p):.0f} bp")

In [ ]:
# Load coordinates
coords_file = '/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/results_repeated/only_successful_seqs.tsv'  # Update path as needed
slice_path = '/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/results_repeated'  # Update path as needed
min_length = 3

print(f"\nLoading coordinates from: {coords_file}")
coord_df = pd.read_csv(coords_file, sep='\t')
print(f"Found {len(coord_df)} sequences")

In [ ]:
stats = analyze_homopolymer_distribution(coord_df, slice_path, min_length)

In [ ]:
print_summary_statistics(stats)

In [ ]:
def plot_homopolymer_analysis(stats, min_length=4, output_prefix='homopolymer_distribution'):
    """
    Create comprehensive plots of homopolymer distribution.
    
    Args:
        stats: Dictionary from analyze_homopolymer_distribution
        min_length: minimum homopolymer length (for titles)
        output_prefix: prefix for output files
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: Histogram of distances between homopolymers
    ax = axes[0, 0]
    if len(stats['distances']) > 0:
        ax.hist(stats['distances'], bins=50, alpha=0.7, color='steelblue', edgecolor='black')
        ax.axvline(np.median(stats['distances']), color='red', linestyle='--', 
                   label=f"Median: {np.median(stats['distances']):.1f} bp")
        ax.axvline(np.mean(stats['distances']), color='orange', linestyle='--',
                   label=f"Mean: {np.mean(stats['distances']):.1f} bp")
        ax.set_xlabel('Distance between consecutive homopolymers (bp)', fontweight='bold')
        ax.set_ylabel('Count', fontweight='bold')
        ax.set_title(f'Inter-Homopolymer Distances (≥{min_length}bp)', fontweight='bold')
        ax.legend()
        ax.grid(alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center')
    
    # Plot 2: Histogram of positions (where in the 2048bp slice)
    ax = axes[0, 1]
    if len(stats['positions']) > 0:
        ax.hist(stats['positions'], bins=50, alpha=0.7, color='forestgreen', edgecolor='black')
        ax.set_xlabel('Position in slice (bp)', fontweight='bold')
        ax.set_ylabel('Count', fontweight='bold')
        ax.set_title('Homopolymer Position Distribution', fontweight='bold')
        ax.grid(alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center')
    
    # Plot 3: Histogram of homopolymer lengths
    ax = axes[1, 0]
    if len(stats['lengths']) > 0:
        length_counts = pd.Series(stats['lengths']).value_counts().sort_index()
        ax.bar(length_counts.index, length_counts.values, alpha=0.7, 
               color='coral', edgecolor='black')
        ax.set_xlabel('Homopolymer length (bp)', fontweight='bold')
        ax.set_ylabel('Count', fontweight='bold')
        ax.set_title('Homopolymer Length Distribution', fontweight='bold')
        ax.grid(alpha=0.3, axis='y')
    else:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center')
    
    # Plot 4: Homopolymers per sequence
    ax = axes[1, 1]
    if len(stats['count_per_seq']) > 0:
        ax.hist(stats['count_per_seq'], bins=range(0, int(stats['count_per_seq'].max()) + 2), 
                alpha=0.7, color='mediumpurple', edgecolor='black')
        ax.axvline(np.median(stats['count_per_seq']), color='red', linestyle='--',
                   label=f"Median: {np.median(stats['count_per_seq']):.1f}")
        ax.axvline(np.mean(stats['count_per_seq']), color='orange', linestyle='--',
                   label=f"Mean: {np.mean(stats['count_per_seq']):.1f}")
        ax.set_xlabel('Number of homopolymers per sequence', fontweight='bold')
        ax.set_ylabel('Count', fontweight='bold')
        ax.set_title('Homopolymers per Sequence', fontweight='bold')
        ax.legend()
        ax.grid(alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center')
    
    plt.tight_layout()
    plt.savefig(f'{output_prefix}.png', dpi=300, bbox_inches='tight')
    print(f"Saved: {output_prefix}.png")
    plt.show()
    
    # Additional plot: G vs C comparison
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Separate by base type
    g_lengths = [stats['lengths'][i] for i in range(len(stats['lengths'])) if stats['bases'][i] == 'G']
    c_lengths = [stats['lengths'][i] for i in range(len(stats['lengths'])) if stats['bases'][i] == 'C']
    
    # Plot G homopolymers
    ax = axes[0]
    if len(g_lengths) > 0:
        length_counts = pd.Series(g_lengths).value_counts().sort_index()
        ax.bar(length_counts.index, length_counts.values, alpha=0.7, 
               color='#2E7D32', edgecolor='black')
        ax.set_xlabel('Length (bp)', fontweight='bold')
        ax.set_ylabel('Count', fontweight='bold')
        ax.set_title(f'G Homopolymers (n={len(g_lengths)})', fontweight='bold')
        ax.grid(alpha=0.3, axis='y')
    
    # Plot C homopolymers
    ax = axes[1]
    if len(c_lengths) > 0:
        length_counts = pd.Series(c_lengths).value_counts().sort_index()
        ax.bar(length_counts.index, length_counts.values, alpha=0.7,
               color='#1565C0', edgecolor='black')
        ax.set_xlabel('Length (bp)', fontweight='bold')
        ax.set_ylabel('Count', fontweight='bold')
        ax.set_title(f'C Homopolymers (n={len(c_lengths)})', fontweight='bold')
        ax.grid(alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(f'{output_prefix}_by_base.png', dpi=300, bbox_inches='tight')
    print(f"Saved: {output_prefix}_by_base.png")
    plt.show()

In [ ]:
plot_homopolymer_analysis(stats, min_length, 'homopolymer_distribution')

In [ ]:
def plot_individual_sequences(coord_df, slice_path, n_sequences=5, min_length=4, seed=42):
    """
    Plot homopolymer positions for individual sequences.
    
    Args:
        coord_df: DataFrame with coordinates
        slice_path: Path to slices
        n_sequences: Number of sequences to plot
        min_length: Minimum homopolymer length
        seed: Random seed for sequence selection
    """
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    
    # Randomly select sequences
    np.random.seed(seed)
    selected_indices = np.random.choice(len(coord_df), size=n_sequences, replace=False)
    
    # Create figure
    fig, axes = plt.subplots(n_sequences, 1, figsize=(14, 2.5 * n_sequences))
    if n_sequences == 1:
        axes = [axes]
    
    for plot_idx, seq_idx in enumerate(selected_indices):
        row = coord_df.iloc[seq_idx]
        chrom = row["chrom"]
        start = row["centered_start"]
        end = row["centered_end"]
        fold = row["fold"]
        
        # Load edited slice
        slice_path_full = f"{slice_path}/fold{fold}/{chrom}_{start}_{end}_slice.pt"
        slice_tensor = torch.load(slice_path_full, weights_only=True, map_location=device)
        sequence = tensor_to_sequence(slice_tensor)
        seq_length = len(sequence)
        
        # Find homopolymers
        homopolymers = find_homopolymers(sequence, min_length)
        
        # Plot
        ax = axes[plot_idx]
        
        # Draw homopolymers as horizontal bars
        for homo in homopolymers:
            color = '#2E7D32' if homo['base'] == 'G' else '#1565C0'
            y_position = 0.5 if homo['base'] == 'G' else -0.5
            
            ax.barh(y_position, homo['length'], left=homo['start'], 
                   height=0.8, color=color, alpha=0.7, edgecolor='black', linewidth=0.5)
        
        # Formatting
        ax.set_xlim(0, seq_length)
        ax.set_ylim(-1.5, 1.5)
        ax.set_yticks([-0.5, 0.5])
        ax.set_yticklabels(['C', 'G'])
        ax.set_xlabel('Position (bp)', fontweight='bold')
        ax.set_title(f'Sequence {seq_idx}: {chrom}:{start}-{end} ({len(homopolymers)} homopolymers)', 
                    fontweight='bold', fontsize=10)
        ax.grid(axis='x', alpha=0.3)
        ax.axhline(0, color='black', linewidth=0.5, linestyle='--', alpha=0.3)
        
        # Add statistics as text
        g_count = sum(1 for h in homopolymers if h['base'] == 'G')
        c_count = sum(1 for h in homopolymers if h['base'] == 'C')
        
        ax.text(0.98, 0.95, f"G: {g_count}  C: {c_count}", 
               transform=ax.transAxes, ha='right', va='top',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
               fontsize=9)
    
    plt.tight_layout()
    # plt.savefig('homopolymer_positions_individual.png', dpi=300, bbox_inches='tight')
    # print(f"Saved: homopolymer_positions_individual.png")
    plt.show()

In [ ]:
def plot_density_heatmap(coord_df, slice_path, n_sequences=20, min_length=4, bin_size=50, seed=42):
    """
    Create a heatmap showing homopolymer density across sequences.
    
    Args:
        coord_df: DataFrame with coordinates
        slice_path: Path to slices
        n_sequences: Number of sequences to include
        min_length: Minimum homopolymer length
        bin_size: Bin size for density calculation (bp)
        seed: Random seed
    """
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    
    # Select sequences
    np.random.seed(seed)
    selected_indices = np.random.choice(len(coord_df), size=min(n_sequences, len(coord_df)), replace=False)
    
    # Determine number of bins
    first_row = coord_df.iloc[selected_indices[0]]
    first_slice = torch.load(f"{slice_path}/fold{first_row['fold']}/{first_row['chrom']}_{first_row['centered_start']}_{first_row['centered_end']}_slice.pt", 
                             weights_only=True, map_location=device)
    seq_length = first_slice.shape[-1]
    n_bins = seq_length // bin_size
    
    # Create density matrix
    density_matrix = np.zeros((len(selected_indices), n_bins))
    
    for i, seq_idx in enumerate(selected_indices):
        row = coord_df.iloc[seq_idx]
        chrom = row["chrom"]
        start = row["centered_start"]
        end = row["centered_end"]
        fold = row["fold"]
        
        # Load edited slice
        slice_path_full = f"{slice_path}/fold{fold}/{chrom}_{start}_{end}_slice.pt"
        slice_tensor = torch.load(slice_path_full, weights_only=True, map_location=device)
        sequence = tensor_to_sequence(slice_tensor)
        
        # Find homopolymers
        homopolymers = find_homopolymers(sequence, min_length)
        
        # Calculate density in each bin
        for homo in homopolymers:
            # Count this homopolymer in all bins it overlaps
            start_bin = homo['start'] // bin_size
            end_bin = min(homo['end'] // bin_size, n_bins - 1)
            for bin_idx in range(start_bin, end_bin + 1):
                density_matrix[i, bin_idx] += 1
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=(14, 8))
    
    im = ax.imshow(density_matrix, aspect='auto', cmap='YlOrRd', interpolation='nearest')
    
    # Formatting
    ax.set_xlabel('Position (bp)', fontweight='bold', fontsize=12)
    ax.set_ylabel('Sequence', fontweight='bold', fontsize=12)
    ax.set_title(f'Homopolymer Density Heatmap (≥{min_length}bp, {bin_size}bp bins)', 
                fontweight='bold', fontsize=14)
    
    # X-axis: show positions in bp
    xtick_positions = np.arange(0, n_bins, max(1, n_bins // 10))
    xtick_labels = [f"{int(pos * bin_size)}" for pos in xtick_positions]
    ax.set_xticks(xtick_positions)
    ax.set_xticklabels(xtick_labels)
    
    # Y-axis: show sequence indices
    ax.set_yticks(range(len(selected_indices)))
    ax.set_yticklabels([f"Seq {idx}" for idx in selected_indices], fontsize=8)
    
    # Colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Homopolymer count per bin', fontweight='bold', fontsize=11)
    
    plt.tight_layout()
    # plt.savefig('homopolymer_density_heatmap.png', dpi=300, bbox_inches='tight')
    # print(f"Saved: homopolymer_density_heatmap.png")
    plt.show()

In [ ]:
def analyze_clustering(coord_df, slice_path, min_length=4, max_sequences=100):
    """
    Quantify clustering by calculating variance in local density.
    High variance = clustered, low variance = uniform.
    """
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    
    variances = []
    
    for idx in range(min(max_sequences, len(coord_df))):
        row = coord_df.iloc[idx]
        chrom = row["chrom"]
        start = row["centered_start"]
        end = row["centered_end"]
        fold = row["fold"]
        
        # Load edited slice
        slice_path_full = f"{slice_path}/fold{fold}/{chrom}_{start}_{end}_slice.pt"
        slice_tensor = torch.load(slice_path_full, weights_only=True, map_location=device)
        sequence = tensor_to_sequence(slice_tensor)
        
        # Find homopolymers
        homopolymers = find_homopolymers(sequence, min_length)
        
        if len(homopolymers) == 0:
            continue
        
        # Calculate local density (count in sliding windows)
        window_size = 100
        step = 50
        densities = []
        
        for window_start in range(0, len(sequence) - window_size, step):
            window_end = window_start + window_size
            count = sum(1 for h in homopolymers 
                       if h['start'] < window_end and h['end'] > window_start)
            densities.append(count)
        
        # Variance measures clustering
        if len(densities) > 1:
            variances.append(np.var(densities))
    
    return np.array(variances)

In [ ]:
plot_individual_sequences(coord_df, slice_path, n_sequences=5, min_length=min_length, seed=42)

In [ ]:
plot_density_heatmap(coord_df, slice_path, n_sequences=20, min_length=min_length, bin_size=50, seed=42)

In [ ]:
variances = analyze_clustering(coord_df, slice_path, min_length=min_length, max_sequences=100)

In [ ]:
print(f"\nClustering analysis (variance in local density):")
print(f"  Mean variance: {np.mean(variances):.2f}")
print(f"  Median variance: {np.median(variances):.2f}")
print(f"  Higher variance = more clustering")

In [ ]:
CTCF_PWM = "/home1/smaruj/IterativeMutagenesis/MA0139.1.meme"

In [ ]:
def read_meme_pwm_as_numpy(filename):
    pwm_list = []  # List to store PWM rows
    
    with open(filename, 'r') as file:
        in_matrix_section = False
        
        for line in file:
            line = line.strip()
            
            # Check if we are reading the PWM matrix
            if line.startswith("letter-probability matrix"):
                in_matrix_section = True  # Start reading matrix data
                continue  # Skip this header line
            
            # If we are in the matrix section, process the rows
            if in_matrix_section and line:
                pwm_row = [float(value) for value in line.split()]  # Parse values
                pwm_list.append(pwm_row)  # Append to the PWM list
            
            # If we encounter a new MOTIF or the end of file, stop matrix reading
            if line.startswith("MOTIF") and in_matrix_section:
                break
    
    # Convert the list to a numpy array
    pwm_array = np.array(pwm_list)
    
    return pwm_array

In [ ]:
pwm_CTCF = read_meme_pwm_as_numpy(CTCF_PWM)
pwm_CTCF_tensor = torch.from_numpy(pwm_CTCF.T).float()
motifs_dict = {"CTCF": pwm_CTCF_tensor}

In [ ]:
pwm_ctcf = np.load("./PWM_with_flanks.npy")

In [ ]:
left_flank_slice = pwm_ctcf[:15, :]  
left_flank_transposed = left_flank_slice.T 
left_flank_tensor = torch.tensor(left_flank_transposed, dtype=torch.float32)

In [ ]:
right_flank_slice = pwm_ctcf[-15:, :]  
right_flank_transposed = right_flank_slice.T 
right_flank_tensor = torch.tensor(right_flank_transposed, dtype=torch.float32)

In [ ]:
motifs_dict = {"CTCF": pwm_CTCF_tensor,
               "left_flank": left_flank_tensor,
               "right_flank": right_flank_tensor}

In [ ]:
from memelite import fimo

In [ ]:
batch_size = 4

orig_num_CTCFs = []
num_CTCFs = []

orig_CTCFs_coord = []
new_CTCFs_coord = []

avg_orig_fimo_scores = []
avg_new_fimo_scores = []

avg_orig_left_fimo_scores = []
avg_new_left_fimo_scores = []

avg_orig_right_fimo_scores = []
avg_new_right_fimo_scores = []

extra_flank = 60

model.eval()
with torch.no_grad():
    for orig_batch, edited_batch in zip(orig_loader, edited_loader):
        
        orig_batch = orig_batch.to(device)
        edited_batch = edited_batch.to(device)
        
        orig_slice = orig_batch[:, :, edit_start-extra_flank:edit_end+extra_flank]
        edited_slice = edited_batch[:, :, edit_start-extra_flank:edit_end+extra_flank]
        
        # original sequences
        ctcf_orig_hits, left_orig_hits, right_orig_hits = fimo(
            motifs=motifs_dict,
            sequences=orig_slice.cpu(),
            threshold=1e-4,
            reverse_complement=True
        )
        
        # Shift coordinates back to original reference
        ctcf_orig_hits["start"] -= extra_flank
        ctcf_orig_hits["end"]   -= extra_flank
        left_orig_hits["start"] -= extra_flank
        left_orig_hits["end"]   -= extra_flank
        right_orig_hits["start"] -= extra_flank
        right_orig_hits["end"]   -= extra_flank
        
        # edited sequences
        ctcf_edit_hits, left_edit_hits, right_edit_hits = fimo(
            motifs=motifs_dict,
            sequences=edited_slice.cpu(),
            threshold=1e-4,
            reverse_complement=True
        )

        # Shift coordinates back to original reference
        ctcf_edit_hits["start"] -= extra_flank
        ctcf_edit_hits["end"]   -= extra_flank
        left_edit_hits["start"] -= extra_flank
        left_edit_hits["end"]   -= extra_flank
        right_edit_hits["start"] -= extra_flank
        right_edit_hits["end"]   -= extra_flank
        
        for seq_idx in range(batch_size):
            # --- ORIGINAL, CTCF ---
            orig_seq_hits = ctcf_orig_hits[ctcf_orig_hits["sequence_name"] == seq_idx]
            if not orig_seq_hits.empty:
                orig_seq_hits = orig_seq_hits.sort_values(by="start")
                orig_num_CTCFs.append(len(orig_seq_hits))
                orig_ctcf_coords = set(zip(orig_seq_hits["start"], orig_seq_hits["end"], orig_seq_hits["strand"]))
                orig_fimo_score_avg = orig_seq_hits["score"].mean()
            else:
                orig_num_CTCFs.append(0)
                orig_ctcf_coords = set()
                orig_fimo_score_avg = 0.0
            orig_CTCFs_coord.append(orig_ctcf_coords)
            avg_orig_fimo_scores.append(orig_fimo_score_avg)
            
            # --- ORIGINAL, LEFT FLANK ---
            orig_left_hits = left_orig_hits[left_orig_hits["sequence_name"] == seq_idx]
            if not orig_left_hits.empty:
                orig_fimo_left_score_avg = orig_left_hits["score"].mean()
            else:
                orig_fimo_left_score_avg = 0.0
            avg_orig_left_fimo_scores.append(orig_fimo_left_score_avg)
                
            # --- ORIGINAL, RIGHT FLANK ---
            orig_right_hits = right_orig_hits[right_orig_hits["sequence_name"] == seq_idx]
            if not orig_right_hits.empty:
                orig_fimo_right_score_avg = orig_right_hits["score"].mean()
            else:
                orig_fimo_right_score_avg = 0.0
            avg_orig_right_fimo_scores.append(orig_fimo_right_score_avg)
            
            # --- EDITED, CTCF ---
            edited_ctcf_seq_hits = ctcf_edit_hits[ctcf_edit_hits["sequence_name"] == seq_idx]
            if not edited_ctcf_seq_hits.empty:
                edited_ctcf_seq_hits = edited_ctcf_seq_hits.sort_values(by="start")
                num_CTCFs.append(len(edited_ctcf_seq_hits))

                # New CTCF sites only
                new_hits = [
                    (start, end, strand, score)
                    for start, end, strand, score in zip(
                        edited_ctcf_seq_hits["start"],
                        edited_ctcf_seq_hits["end"],
                        edited_ctcf_seq_hits["strand"],
                        edited_ctcf_seq_hits["score"]
                    )
                    if (start, end, strand) not in orig_ctcf_coords
                ]
                
                df_new_hits = pd.DataFrame(new_hits, columns=["start", "end", "strand", "score"])
                
                if not df_new_hits.empty:
                    df_new_hits = df_new_hits.sort_values(by="start")
                    new_ctcf_coords = set(zip(df_new_hits["start"], df_new_hits["end"], df_new_hits["strand"]))
                    new_fimo_scores = df_new_hits["score"].mean()
                else:
                    new_ctcf_coords = set()
                    new_fimo_scores = 0.0
                new_CTCFs_coord.append(new_ctcf_coords)
                avg_new_fimo_scores.append(new_fimo_scores)
        
            else:
                num_CTCFs.append(0)
                new_CTCFs_coord.append(set())
                avg_new_fimo_scores.append(0.0)   
            
            # --- ORIGINAL, LEFT FLANK ---
            edit_left_hits = left_edit_hits[left_edit_hits["sequence_name"] == seq_idx]
            if not edit_left_hits.empty:
                edit_fimo_left_score_avg = edit_left_hits["score"].mean()
            else:
                edit_fimo_left_score_avg = 0.0
            avg_new_left_fimo_scores.append(edit_fimo_left_score_avg)
                
            # --- ORIGINAL, RIGHT FLANK ---
            edit_right_hits = right_edit_hits[right_edit_hits["sequence_name"] == seq_idx]
            if not edit_right_hits.empty:
                edit_fimo_right_score_avg = edit_right_hits["score"].mean()
            else:
                edit_fimo_right_score_avg = 0.0
            avg_new_right_fimo_scores.append(edit_fimo_right_score_avg)        
                   

In [ ]:
num_CTCFs

In [ ]:
df["init_CTCFs_num"] = orig_num_CTCFs[:len(df)]
df["CTCFs_num"] = num_CTCFs[:len(df)]

In [ ]:
df[]

In [ ]:
new_CTCFs_coord

In [ ]:
orig_preds_all = torch.cat(preds_all_orig, dim=0)
edited_preds_all = torch.cat(preds_all_reversed, dim=0)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
df["SCD"] = scd_values
df["URQ_result"] = urq_mean_values
df["URQ_target"] = target_urq_mean_values
df["URQ_init"] = og_urq_mean_values
df["num_edits"] = edit_counts
df["GC_seq"] = seq_GC_content
df["GC_slice"] = slice_GC_content
df["GC_slice_edited"] = edited_GC_content
# df["flat_GC_content"] = flat_GC_content

In [ ]:
df["init_CTCFs_num"] = orig_num_CTCFs[:len(df)]
df["CTCFs_num"] = num_CTCFs[:len(df)]
df["avg_orig_fimo_scores"] = avg_orig_fimo_scores[:len(df)]
df["avg_new_fimo_scores"] = avg_new_fimo_scores[:len(df)]

df["orig_CTCFs_coord"] = orig_CTCFs_coord[:len(df)]
df["new_CTCFs_coord"] = new_CTCFs_coord[:len(df)]

df["avg_orig_left_fimo_scores"] = avg_orig_left_fimo_scores[:len(df)]
df["avg_new_left_fimo_scores"] = avg_new_left_fimo_scores[:len(df)]

df["avg_orig_right_fimo_scores"] = avg_orig_right_fimo_scores[:len(df)]
df["avg_new_right_fimo_scores"] = avg_new_right_fimo_scores[:len(df)]

# df["FIMO_sum"] = sum_FIMO[:len(df)]
# df["FIMO_max"] = max_FIMO[:len(df)]
# df["orientation"] = strand_strings[:len(df)]
# df["positions"] = positions[:len(df)]

In [ ]:
df["orig_CTCFs_coord_shuf"] = orig_CTCFs_coord[:len(df)]
df["new_CTCFs_coord_shuf"] = new_CTCFs_coord[:len(df)]

In [ ]:
len(new_CTCFs_coord)

In [ ]:
new_CTCFs_coord

In [ ]:
new_CTCFs_coord

In [ ]:
df[['orig_CTCFs_coord', 'orig_CTCFs_coord_shuf']]

In [ ]:
df[['new_CTCFs_coord', 'new_CTCFs_coord_shuf']]

In [ ]:
df.columns

In [ ]:
df

In [ ]:
df.to_csv(f"/scratch1/smaruj/suppressing_CTCFs/results/fold{FOLD}_with_positions_steps_results.tsv", sep="\t", index=False)
# df.to_csv(f"/scratch1/smaruj/suppressing_CTCFs/results_control/fold{FOLD}_with_positions_steps_results.tsv", sep="\t", index=False)

In [ ]:
batch_size = 4
flank_size = 15
ctcf_ohe_seqs = []  # list to collect OHE motif+flank arrays
ctcf_ohe_scores = [] # FIMO

model.eval()
with torch.no_grad():
    for edited_batch in edited_loader:
        
        edited_batch = edited_batch.to(device)
        edited_slice = edited_batch[:, :, edit_start:edit_end]  # shape: [B, 4, L]
        
        hits = fimo.fimo(
            motifs=motifs_dict,
            sequences=edited_slice,
            threshold=1e-4,
            reverse_complement=True
        )[0]
        
        for seq_idx in range(batch_size):
            seq_hits = hits[hits["sequence_name"] == seq_idx]
            
            if not seq_hits.empty:
                for _, row in seq_hits.iterrows():
                    start = int(row["start"]) - flank_size
                    end = int(row["end"]) + flank_size
                    strand = row["strand"]
                    
                    # Make sure bounds are within slice
                    if start < 0 or end > edited_slice.shape[-1]:
                        continue
                    
                    subseq = edited_slice[seq_idx, :, start:end]  # shape: [4, motif_len+30]
                    
                    if strand == '-':
                        # Reverse sequence
                        subseq = torch.flip(subseq, dims=[-1])  # reverse along the sequence axis

                        # Complement the one-hot rows: A<->T, C<->G
                        complement_map = torch.tensor([3, 2, 1, 0], device=subseq.device)  # old row i becomes new row complement_map[i]
                        subseq = subseq[complement_map, :]
    
                    ctcf_ohe_seqs.append(subseq.cpu().numpy())  # store as numpy array
                    ctcf_ohe_scores.append(row["score"])

In [ ]:
top_indices = np.argsort(ctcf_ohe_scores)[-20:][::-1]  # descending order

In [ ]:
# Extract the top sequences
top_ctcf_ohe = [ctcf_ohe_seqs[i] for i in top_indices]
top_scores = [ctcf_ohe_scores[i] for i in top_indices]

In [ ]:
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, leaves_list

In [ ]:
# Assume top_ctcf_ohe is a list of 20 np.arrays with shape (4, 49)
# Stack them into a single array of shape (20, 4, 49)
seq_array = np.stack(top_ctcf_ohe)  # shape: [20, 4, 49]

# Convert from one-hot to integer-encoded sequences (A=0, C=1, G=2, T=3)
int_encoded = np.argmax(seq_array, axis=1)  # shape: [20, 49]

# Compute pairwise Hamming distances
distance_matrix = squareform(pdist(int_encoded, metric='hamming'))

# Use hierarchical clustering to order sequences
order = leaves_list(linkage(distance_matrix))

# Reorder the sequences
ordered_seqs = int_encoded[order]  # shape: [20, 49]

In [ ]:
# Map base integers to RGB colors
color_map = {
    0: [0.0, 1.0, 0.0],   # A - green
    1: [0.0, 0.0, 1.0],   # C - blue
    2: [1.0, 1.0, 0.0],   # G - yellow
    3: [1.0, 0.0, 0.0],   # T - red
}
rgb_array = np.array([[color_map[base] for base in row] for row in ordered_seqs])  # shape: [20, 49, 3]

In [ ]:
# Plotting
plt.figure(figsize=(12, 6))
plt.imshow(rgb_array, aspect='auto')
plt.title('Ordered CTCF Motifs (by Hamming Distance)')
plt.xlabel('Position')
plt.ylabel('Sequence Index')
plt.xticks(ticks=np.linspace(0, 48, 7), labels=[f'{int(x)}' for x in np.linspace(1, 49, 7)])
plt.yticks(ticks=np.arange(20), labels=[str(i + 1) for i in range(20)])
plt.tight_layout()
plt.show()

### SCD

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(df["SCD"], bins=30, color='steelblue', edgecolor='black')
plt.xlabel("SCD")
plt.ylabel("Count")
plt.title("Distribution of SCD values")
plt.tight_layout()
plt.show()

In [ ]:
# Make sure scd_values is a list or Series of same length as pivoted
assert len(scd_values) == len(df), "Length mismatch!"

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

state_labels = ["active", "neutral", "repressive"]

for i, label in enumerate(state_labels):
    axes[i].scatter(scd_values, df[f"{label}_fraction"], alpha=0.7)
    axes[i].set_title(f"{label.capitalize()} Fraction vs SCD")
    axes[i].set_xlabel("SCD Value")
    if i == 0:
        axes[i].set_ylabel("Fraction")

plt.tight_layout()
plt.show()

### URQ mean

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(urq_mean_values, bins=30, color='steelblue', edgecolor='black')
plt.xlabel("URQ mean")
plt.ylabel("Count")
plt.title("Distribution of URQ mean values")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(target_urq_mean_values, bins=30, color='steelblue', edgecolor='black')
plt.xlabel("Target URQ mean")
plt.ylabel("Count")
plt.title("Distribution of Target URQ mean values")
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import pearsonr

In [ ]:
# Compute Pearson R
r_val, p_val = pearsonr(urq_mean_values, target_urq_mean_values)

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(urq_mean_values, target_urq_mean_values, alpha=0.6, edgecolors='k')

plt.xlabel("Edited URQ Mean")
plt.ylabel("Target URQ Mean")
plt.title("URQ Mean: Edited vs. Target")

# Annotate with Pearson R
plt.text(0.05, 0.95, f"Pearson R = {r_val:.3f}", transform=plt.gca().transAxes,
         fontsize=12, verticalalignment='top')

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Make sure scd_values is a list or Series of same length as pivoted
assert len(urq_mean_values) == len(df), "Length mismatch!"

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

state_labels = ["active", "neutral", "repressive"]

for i, label in enumerate(state_labels):
    axes[i].scatter(urq_mean_values, df[f"{label}_fraction"], alpha=0.7)
    axes[i].set_title(f"{label.capitalize()} Fraction vs URQ mean")
    axes[i].set_xlabel("URQ mean")
    if i == 0:
        axes[i].set_ylabel("Fraction")

plt.tight_layout()
plt.show()

### edit count

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(edit_counts, bins=30, color='steelblue', edgecolor='black')
plt.xlabel("Number of edits / 2048bp-long bin")
plt.ylabel("Count")
plt.title("Distribution of edits number")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(df["last_accepted_step_query"], bins=30, color='steelblue', edgecolor='black')
plt.xlabel("Last step with accepted edits")
plt.ylabel("Count")
plt.title("Distribution of last step with accepted edits")
plt.tight_layout()
plt.show()

### GC content

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(edit_counts, seq_GC_content, alpha=0.6, edgecolors='k')

plt.xlabel("number of edits")
plt.ylabel("Seq GC content")
plt.title("number of edits vs. Seq GC content")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(df["last_accepted_step_query"], seq_GC_content, alpha=0.6, edgecolors='k')

plt.xlabel("last step with accepted edit")
plt.ylabel("Seq GC content")
plt.title("last step with accepted edit vs. Seq GC content")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(urq_mean_values, seq_GC_content, alpha=0.6, edgecolors='k')

plt.xlabel("URQ mean")
plt.ylabel("Seq GC content")
plt.title("URQ mean vs. Seq GC content")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# df.to_csv(f"/scratch1/smaruj/genomic_insertion_loci/fold{FOLD}_{TARGET_C}_all_data.tsv", sep="\t", index=False)

In [ ]:
# Helper function to set diagonal elements to a specific value
def set_diag(matrix, value, k):
    # Explicitly set the diagonal to 'value' (in this case, np.nan) for each k
    rows, cols = matrix.shape
    for i in range(rows):
        if 0 <= i + k < cols:
            matrix[i, i + k] = value

def from_upper_triu(vector_repr, matrix_len, num_diags):
    # Ensure vector_repr is a NumPy array (if it's a PyTorch tensor, convert it)
    if isinstance(vector_repr, torch.Tensor):
        vector_repr = vector_repr.detach().flatten().cpu().numpy()  # Flatten and convert to NumPy array

    # Initialize a zero matrix of shape (matrix_len, matrix_len)
    z = np.zeros((matrix_len, matrix_len))

    # Get the indices for the upper triangular matrix
    triu_tup = np.triu_indices(matrix_len, num_diags)

    # Assign the values from the vector_repr to the upper triangular part of the matrix
    z[triu_tup] = vector_repr

    # Set the diagonals specified by num_diags to np.nan
    for i in range(-num_diags + 1, num_diags):
        set_diag(z, np.nan, i)

    # Ensure the matrix is symmetric
    return z + z.T

In [ ]:
i = 12

In [ ]:
df['URQ_homopolymers']

In [ ]:
plt.figure(figsize=(8, 8))
matrix = from_upper_triu(orig_preds_all[i,:], matrix_len=512, num_diags=2)
plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
# plt.colorbar()
# plt.savefig("before_optimization_map.svg", format='svg')
plt.show()


In [ ]:
plt.figure(figsize=(8, 8))
matrix = from_upper_triu(edited_preds_all[i,:], matrix_len=512, num_diags=2)
plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
# plt.colorbar()
# plt.savefig("after_optimization_map.svg", format='svg')
plt.show()


In [ ]:
plt.figure(figsize=(8, 8))
matrix = from_upper_triu(edited_preds_all[i,:] - orig_preds_all[i,:], matrix_len=512, num_diags=2)
plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
# plt.colorbar()
# plt.savefig("difference_map.svg", format='svg')
plt.show()


In [ ]:
plt.figure(figsize=(8, 8))
matrix = from_upper_triu(targets_all[i,:], matrix_len=512, num_diags=2)
plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
# plt.colorbar()
# plt.savefig("target_map.svg", format='svg')
plt.show()

In [ ]:
for i in range(6):
    print(i)
    matrix = from_upper_triu(orig_preds_all[i,:], matrix_len=512, num_diags=2)
    
    plt.figure(figsize=(5, 5))
    plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
    plt.colorbar()
    plt.show()

In [ ]:
for i in range(22, 26):
    print(i)
    matrix = from_upper_triu(edited_preds_all[i,:], matrix_len=512, num_diags=2)
    
    plt.figure(figsize=(5, 5))
    plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
    plt.colorbar()
    plt.show()

In [ ]:
for i in range(22, 26):
    print(i)
    matrix = from_upper_triu(edited_preds_all[i,:] - orig_preds_all[i,:], matrix_len=512, num_diags=2)
    
    plt.figure(figsize=(5, 5))
    plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
    plt.colorbar()
    plt.show()

In [ ]:
for i in range(22, 26):
    print(i)
    matrix = from_upper_triu(targets_all[i,:], matrix_len=512, num_diags=2)
    
    plt.figure(figsize=(5, 5))
    plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
    plt.colorbar()
    plt.show()

In [ ]:
df